# Analisi Visuale degli Appalti Pubblici in Portogallo (PPP)
## Caso di Studio per l'Esame di Visualizzazione Dati e Visual Analytics (VDVAR)

**Studente:** Domenico Lacavalla
**Data:** 05/11/2025

---

### 1. Obiettivi e Contesto Teoretico

Questo notebook presenta un'analisi end-to-end del dataset "Public Procurement in Portugal" (PPP). L'obiettivo non è solo descrivere i dati, ma applicare i principi di **Information Visualization (InfoVis)** e **Visual Analytics (VA)** per estrarre conoscenza e "amplificare la cognizione", come definito da Shneiderman.

Il caso di studio segue l'intero processo di generazione di *insight*, dai dati grezzi alla conoscenza:
1.  **Dati (Raw Data):** Caricamento e ispezione.
2.  **Pre-processing & Analysis:** Applicazione del **Mantra di Visual Analytics di Keim** (*"Analyze first, Show the Important..."*). Eseguiamo prima un'analisi automatizzata (pulizia, feature engineering, clustering) per gestire la complessità e preparare i dati.
3.  **Visual Mapping:** Scelta deliberata delle tecniche di visualizzazione (il *Design Space*) in base al tipo di dati (temporali, n-D, gerarchici) e ai **canali visivi** più efficaci (es. posizione, lunghezza, colore).
4.  **Storytelling & Interazione:** Presentazione dei risultati seguendo il **Mantra della Ricerca di Informazioni di Shneiderman** (*"Overview first, zoom and filter, then details-on-demand"*).

### 2. Architettura del Notebook

L'architettura del codice è stata progettata per essere:
* **Modulare:** Separazione netta tra configurazione (`Config`), elaborazione (`DataCleaner`, `FeatureEngineer`) e visualizzazione (`BasePlotter`, `...Analyzer`).
* **Riproducibile:** Uso di percorsi relativi e configurazioni centralizzate per garantire che l'analisi sia verificabile.
* **Manutenibile:** Adozione del principio DRY (Don't Repeat Yourself) tramite classi base.

### 3. Setup dell'Ambiente e Configurazione

Per garantire la robustezza e la **riproducibilità** dell'analisi, tutte le costanti (percorsi file, nomi delle colonne, parametri grafici) sono centralizzate in una singola classe di configurazione (`Config`).

Questo approccio agisce da "Single Source of Truth", evitando valori *hardcoded* e facilitando la manutenzione. È un passo fondamentale di ingegneria del software che supporta l'affidabilità dell'intera pipeline di Visual Analytics.

In [ ]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
from IPython.display import IFrame, display
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter
import plotly.express as px
import plotly.io as pio
pio.templates.default = "plotly_white"
import warnings
warnings.filterwarnings("ignore")

class Config:
    """
    SINGLE SOURCE OF TRUTH.
    Centralizza tutte le configurazioni, percorsi e costanti del progetto.
    """
    # --- 1. Percorsi File (usando pathlib per compatibilità OS) ---
    BASE_DIR = Path.cwd()
    RAW_DATA = BASE_DIR / 'Datasets' / 'PPPData_EN_1.0.xlsx'
    CLEANED_DATA = BASE_DIR / 'Datasets' / 'PPPData_EN_cleaned_2.csv'
    GEOJSON = BASE_DIR / 'Datasets' / 'portugal_districts.geojson'
    PLOTS_DIR = BASE_DIR / 'plots_2'

    # --- 2. Nomi Colonne Chiave (per evitare typo nel codice) ---
    # Originali
    COL_ID = 'ID'
    COL_PRICE = 'Base Bid Price (€)'
    COL_DEADLINE = 'Execution deadline (days)'
    COL_DISTRICT = 'District'
    COL_YEAR = 'Signing Year'
    COL_DATE_SIGN = 'Signing date'
    COL_DATE_CLOSE = 'Closing date'
    COL_AWARD = 'Award criteria class'
    COL_CPVS = 'Cpvs Designation'
    
    # Generate/Derivate
    COL_PRICE_DAY = 'Price per Day'
    COL_DIFF_DATES = 'Days between close and signing'

    # Colonne da rimuovere perché ridondanti, vuote o non rilevanti per questa analisi
    DROP_COLS = [
        'Count', 'ID', 'Short Description1', 'Country', 'Award criteria',
        'Involves joint procurement (with several entities) (T/F)',
        'Awarded by a central purchasing body (T/F)',
        'Conclusion of a framework agreement (T/F)', 'Electronic auction (T/F)',
        'Negotiation phase (T/F)', 'Contracting by lots (T/F)', 'Collateral',
        'Contract end type', 'Justification for price change', 'Justification for deadline change'
    ]

    # Colonne che NON devono avere valori nulli per garantire un'analisi minima valida
    CRITICAL_COLS = [
        'Publication Year', 
        'Municipality', 
        'Base Bid Price (€)'
    ]
    
    # --- 3. Stile Visualizzazioni ---
    PALETTE = "viridis"
    COLOR_PRIMARY = "#3498DB"
    COLOR_SECONDARY = "#E74C3C"
    FIG_SIZE_STD = (12, 8)

    @staticmethod
    def setup():
        """Crea le directory necessarie se non esistono."""
        Config.PLOTS_DIR.mkdir(parents=True, exist_ok=True)
        print(f"Setup completato. Directory grafici: {Config.PLOTS_DIR}")

Config.setup()

### 4. Infrastruttura Software: `BasePlotter`

Per evitare la duplicazione del codice e garantire la coerenza visiva, è stata implementata una classe base `BasePlotter`.

**Motivazione Teorica (Design Space):**
Questa classe non è solo un'utilità di codice (principio DRY), ma è il nostro strumento per governare il **Design Space**. Centralizzando metodi come `_save` e `_format_currency`, imponiamo una "grammatica" visiva uniforme.

Essa standardizza:
1.  **Canali Visivi (Channels):** Assicura che la *palette* di colori e lo stile (dimensione font, sfondo) siano coerenti, riducendo il carico cognitivo dell'utente.
2.  **Substrato (Substrate):** Prepara lo spazio 2D per i grafici futuri.
3.  **Leggibilità:** Il metodo `_format_currency` (es. `€1M` invece di `1000000`) migliora la **leggibilità visiva**. Questo si collega allo studio "Beyond Memorability" (Borkin et al.), che evidenzia l'importanza della **ridondanza dei dati** (l'etichetta testuale rinforza il dato visivo) per migliorare la comprensione e il richiamo.

In [ ]:
class BasePlotter:
    """
    Classe genitore per tutte le visualizzazioni.
    Fornisce metodi di utilità condivisi per stile, formattazione e salvataggio.
    """
    def __init__(self):
        # Imposta il tema globale una volta per tutte
        sns.set_theme(style="whitegrid", context="talk", palette=Config.PALETTE)
        plt.rcParams['figure.figsize'] = Config.FIG_SIZE_STD
        plt.rcParams['axes.titleweight'] = 'bold'
        plt.rcParams['axes.titlesize'] = 16

    def _save(self, fig, filename: str):
        """Salva la figura in PNG (per report) gestendo il layout."""
        try: fig.tight_layout()
        except: pass
            
        path = Config.PLOTS_DIR / filename
        fig.savefig(path, dpi=150, bbox_inches='tight')
        print(f"Grafico salvato: {path.name}")
        plt.close(fig) # Chiude per liberare memoria

    def _format_currency(self, ax, axis='y', scale='M'):
        """
        Formatta gli assi numerici in formato valuta leggibile (€).
        scale: 'K' (migliaia), 'M' (milioni) o 'auto'.
        """
        def formatter(x, pos):
            if scale == 'auto':
                if x >= 1e9: return f'€{x*1e-9:.1f}B'
                if x >= 1e6: return f'€{x*1e-6:.1f}M'
                if x >= 1e3: return f'€{x*1e-3:.0f}K'
                return f'€{x:.0f}'
            elif scale == 'M': return f'€{x/1e6:.1f}M'
            elif scale == 'K': return f'€{x/1e3:.0f}K'
            return f'€{x:.0f}'

        func_fmt = FuncFormatter(formatter)
        if axis == 'y': ax.yaxis.set_major_formatter(func_fmt)
        else: ax.xaxis.set_major_formatter(func_fmt)

### 5. Fase 1: Data Loading & Ispezione Preliminare

Iniziamo il processo di Visual Analytics partendo dalla materia prima: i **Dati**. In questa fase, carichiamo il dataset grezzo e conduciamo una prima ispezione della sua integrità (valori nulli, tipi di dato).

**Scelta Tecnica:** Utilizziamo una classe dedicata `DataLoader` per incapsulare la logica di lettura. Questo rende l'analisi agnostica rispetto al formato del file sorgente (CSV, Excel) e migliora la modularità.

In [ ]:
class DataLoader:

    @staticmethod
    def load_raw() -> pd.DataFrame:
        path = Config.RAW_DATA
        print(f"Caricamento dati da: {path.name}...")
        try:
            if path.suffix in ['.xlsx', '.xls']:
                df = pd.read_excel(path)
            elif path.suffix == '.csv':
                df = pd.read_csv(path)
            else:
                raise ValueError("Formato non supportato")

            print(f"Dataset caricato: {df.shape[0]:,} righe, {df.shape[1]} colonne.")
            return df
        except Exception as e:
            print(f"Errore caricamento: {e}")
            return pd.DataFrame()

raw_df = DataLoader.load_raw()

print("\nInfo Dataset Grezzo:")
raw_df.info(memory_usage='deep')

### 5.1 Ispezione Visiva: Valori Mancanti

Prima di qualsiasi pulizia, è fondamentale *visualizzare* l'estensione dei dati mancanti. Le sole statistiche di riepilogo (come un `df.info()`) possono essere fuorvianti. Questo è il principio fondamentale dimostrato dall'**Anscombe's Quartet** e dal **Datasaurus Dozen**: set di dati con statistiche identiche possono avere strutture visive radicalmente diverse.

**Criteri di Scelta Tecnica (Bar Chart Orizzontale):**
Per questa analisi di integrità, un grafico a barre orizzontali è la scelta ottimale.
1.  **Canale Visivo Efficace:** Questa tecnica mappa un dato quantitativo (la percentuale di `null`) al canale visivo più efficace per il confronto: la **Posizione su un asse comune** (l'asse X) e la **Lunghezza**. Questa è in cima alla gerarchia dei canali visivi per i dati quantitativi.
2.  **Elaborazione Preattentiva:** L'uso della lunghezza, un potente **attributo preattentivo**, permette al nostro sistema percettivo di identificare *istantaneamente* le colonne più problematiche (le barre più lunghe) senza dover leggere ogni singolo valore.
3.  **Leggibilità:** L'orientamento orizzontale facilita la lettura delle etichette (nomi delle colonne) sull'asse Y.

In [ ]:
class IntegrityAnalyzer(BasePlotter):
    """Specializzata nell'analisi della qualità dei dati (es. missing values)."""
    
    def plot_missing_values(self, df: pd.DataFrame, title_suffix="") -> None:
        # Calcolo percentuali
        missing = df.isnull().mean() * 100
        missing = missing[missing > 0].sort_values(ascending=True)

        if missing.empty: print("Nessun valore mancante trovato!"); return

        # Creazione Plot
        fig, ax = plt.subplots(figsize=(10, max(6, len(missing) * 0.3)))
        
        # Usa la palette definita in Config
        bars = ax.barh(missing.index, missing.values, color=Config.COLOR_PRIMARY, alpha=0.8)
        
        # Styling
        ax.set_title('Analisi Integrità: Percentuale Valori Mancanti per Colonna', pad=20)
        ax.set_xlabel('Percentuale Mancante (%)')
        ax.set_xlim(0, 100)
        
        # Aggiunta etichette valore sulle barre
        for i, v in enumerate(missing.values): ax.text(v + 1, i, f'{v:.1f}%', va='center', fontsize=10, color='#2C3E50')

        sns.despine()
        plt.show(fig)
        self._save(fig, f'01_missing_values_integrity_check_{title_suffix}.png')
        
integrity_checker = IntegrityAnalyzer()
integrity_checker.plot_missing_values(raw_df, "raw")

### 5.2 Definizione delle Colonne Critiche

Sulla base dell'analisi visiva precedente (che ha evidenziato colonne con molti *missing*) e della *domain knowledge* (comprensione del problema), definiamo formalmente nella `Config` le colonne da rimuovere (`DROP_COLS`) e quelle essenziali per l'analisi (`CRITICAL_COLS`).

Questo è un passo cruciale del **"Analyze first"** (Mantra di Keim): usiamo l'ispezione visiva per informare le nostre regole di pulizia automatizzata.

### 6. Data Cleaning (Mantra di Keim: "Analyze First")

Questa fase implementa la logica di pulizia e trasformazione. La classe `DataCleaner` incapsula tutte le operazioni.

**Contesto Teorico (Keim's VA Mantra):**
Con un dataset di grandi dimensioni, un "Overview" iniziale (come da mantra di Shneiderman) è spesso impossibile o inutile a causa del *noise*. Applichiamo quindi il mantra di Keim: **"Analyze first, Show the Important..."**.

La classe `DataCleaner` *è* il nostro "Analyze first": esegue un'analisi e una pulizia automatizzata (rimozione di colonne irrilevanti, gestione di tipi di dato eterogenei, rimozione di valori nulli critici) per produrre un sottoinsieme di dati pulito e affidabile, che sarà poi "mostrato" nelle fasi successive.

In [ ]:
class DataCleaner:
    """
    Gestisce la pulizia e la standardizzazione del DataFrame.
    Non effettua feature engineering, solo pulizia dei dati esistenti.
    """

    def __init__(self, df: pd.DataFrame) -> None:
        self.df = df.copy()

    def clean_all(self) -> pd.DataFrame:
        """Esegue la pipeline completa di pulizia."""
        self._drop_redundant_columns()
        self._standardize_data_types()
        self._fix_specific_inconsistencies()
        self._remove_critical_missing()
        
        print(f"Pipeline di pulizia completata. Dimensioni finali: {self.df.shape}")
        return self.df

    def _drop_redundant_columns(self) -> None:
        """Rimuove le colonne definite in Config.DROP_COLS."""
        initial_cols = self.df.shape[1]
        self.df.drop(columns=[c for c in Config.DROP_COLS if c in self.df.columns], inplace=True)
        print(f"Colonne rimosse: {initial_cols - self.df.shape[1]}")

    def _standardize_data_types(self) -> None:
        """Normalizza i formati (es. booleani eterogenei, stringhe numeriche)."""
        # Standardizzazione Environmental criteria
        if 'Environmental criteria (T/F)' in self.df.columns:
            self.df['Environmental criteria (T/F)'] = (
                pd.to_numeric(self.df['Environmental criteria (T/F)'], errors='coerce')
                .fillna(0)
                .astype(int)
            )

        # Standardizzazione EU Journal publication
        if 'Published in the EU journal' in self.df.columns:
            mapping = {
                False: 0, 'False': 0, 0: 0, '0': 0,
                True: 1, 'True': 1, 'TRUE ': 1, 1: 1, '1': 1
            }
            self.df['Published in the EU journal'] = self.df['Published in the EU journal'].map(mapping).fillna(0).astype(int)

        # Pulizia stringhe Distretto
        if Config.COL_DISTRICT in self.df.columns: self.df[Config.COL_DISTRICT] = self.df[Config.COL_DISTRICT].astype(str).str.strip()

    def _fix_specific_inconsistencies(self) -> None:
        """Corregge errori noti specifici del dataset (business logic)."""
        # Rimozione incoerenze note nei codici distretto per Beja e Faro
        if 'District Code' in self.df.columns and Config.COL_DISTRICT in self.df.columns:
            mask_beja_error = (self.df[Config.COL_DISTRICT] == 'Beja') & (self.df['District Code'] == 13)
            mask_faro_error = (self.df[Config.COL_DISTRICT] == 'Faro') & (self.df['District Code'] == 13)
            
            rows_to_drop = self.df[mask_beja_error | mask_faro_error].index
            self.df.drop(rows_to_drop, inplace=True)
            self.df.drop(columns=['District Code'], inplace=True, errors='ignore')
            
            if len(rows_to_drop) > 0: print(f"Rimosse {len(rows_to_drop)} righe con incoerenze Distretto/Codice.")

    def _remove_critical_missing(self) -> None:
        """Rimuove righe che non hanno dati sufficienti per l'analisi base."""
        initial_rows = len(self.df)
        # Verifica quali colonne critiche esistono effettivamente nel df
        existing_critical = [col for col in Config.CRITICAL_COLS if col in self.df.columns]
        self.df.dropna(subset=existing_critical, inplace=True)
        dropped = initial_rows - len(self.df)
        if dropped > 0: print(f"Rimosse {dropped} righe con valori mancanti in campi critici {existing_critical}.")

# --- ESECUZIONE CLEANING ---
cleaner = DataCleaner(raw_df)
cleaned_df = cleaner.clean_all()

### 6.1 Verifica Integrità Post-Pulizia

L'analisi visiva non è un processo lineare, ma un *ciclo iterativo*. Dopo aver eseguito la pulizia automatizzata, rieseguiamo l'analisi visiva dei valori mancanti.

Questo *feedback loop* è fondamentale per:
1.  **Validare:** Confermare che le operazioni di pulizia (es. `_remove_critical_missing`) abbiano avuto successo.
2.  **Scoprire:** Identificare eventuali problemi residui (es. colonne non critiche ancora con molti *missing*) che potrebbero richiedere un'imputazione o un'analisi più approfondita.

In [ ]:
integrity_checker.plot_missing_values(cleaned_df, "cleaned")

### 7. Feature Engineering

Il Feature Engineering è il processo di trasformazione dei dati grezzi in *feature* che ne aumentano il potenziale esplicativo. È un passo fondamentale che precede il **Visual Mapping**.

Dati grezzi (come stringhe di date o descrizioni testuali) sono spesso **dati astratti** che, secondo la distinzione **InfoVis vs. SciVis**, non hanno una mappatura geometrica o fisica ovvia. Il nostro compito è trasformarli in dimensioni analizzabili:
1.  **Dati Temporali:** Convertiamo stringhe (es. '05/11/2025') in oggetti `datetime` e ne estraiamo componenti (Anno, Mese). Calcoliamo anche metriche derivate (es. `Days between close and signing`) che possono rivelare *pattern* o inefficienze.
2.  **Dati Finanziari (n-D):** Creiamo metriche normalizzate (es. 'Price per Day'). Questo è essenziale per confrontare contratti con durate diverse, altrimenti un confronto basato solo sul prezzo totale sarebbe fuorviante.
3.  **Dati Testuali (1D-Lineari):** Come vedremo, trasformeremo il testo libero (dati 1D) in *feature* strutturate usando l'NLP.

In [ ]:
class FeatureEngineer:
    def __init__(self, df: pd.DataFrame):
        self.df = df.copy()

    def engineer_all(self) -> pd.DataFrame:
        self._engineer_dates()
        self._engineer_financials()
        print(f"Feature Engineering completato. Nuove dimensioni: {self.df.shape}")
        return self.df

    def _engineer_dates(self):
        date_cols = [Config.COL_DATE_SIGN, Config.COL_DATE_CLOSE]
        # Aggiungiamo 'Publication date' se esiste, anche se non è in Config
        if 'Publication date' in self.df.columns:
             date_cols.append('Publication date')

        for col in date_cols:
            if col not in self.df.columns: continue
            self.df[col] = pd.to_datetime(self.df[col], errors='coerce', dayfirst=True, infer_datetime_format=True)
            
            # Estrae anno e mese
            year_col = f"{col.split()[0]} Year"
            month_col = f"{col.split()[0]} Month"
            self.df[year_col] = self.df[col].dt.year
            self.df[month_col] = self.df[col].dt.month

        # Calcolo differenza giorni
        if all(c in self.df.columns for c in [Config.COL_DATE_CLOSE, Config.COL_DATE_SIGN]):
            self.df[Config.COL_DIFF_DATES] = (self.df[Config.COL_DATE_CLOSE] - self.df[Config.COL_DATE_SIGN]).dt.days

    def _engineer_financials(self):
        if Config.COL_PRICE in self.df.columns and Config.COL_DEADLINE in self.df.columns:
            safe_deadline = self.df[Config.COL_DEADLINE].replace(0, np.nan)
            self.df[Config.COL_PRICE_DAY] = self.df[Config.COL_PRICE] / safe_deadline
            self.df[Config.COL_PRICE_DAY].replace([np.inf, -np.inf], np.nan, inplace=True)

engineer = FeatureEngineer(cleaned_df)
processed_df = engineer.engineer_all()

### 7.1 Feature Engineering Testuale (NLP)

Le descrizioni dei contratti (CPV) sono **dati 1D-Lineari** non strutturati. Per analizzarli visivamente, dobbiamo prima trasformarli in dati **multidimensionali (n-D)**.

**Perché TF-IDF?**
Utilizziamo **TF-IDF (Term Frequency-Inverse Document Frequency)**. A differenza di un semplice conteggio, TF-IDF penalizza le parole troppo comuni (che appaiono in tutti i contratti e quindi non discriminano) ed esalta quelle specifiche di pochi contratti.

Questo è un altro esempio di **"Analyze First"**: usiamo un modello algoritmico per estrarre *feature* latenti (le *keyword* tematiche) dal testo, che verranno poi usate come dimensioni per l'analisi visiva (es. in grafici a barre o mappe semantiche). Questo approccio è concettualmente simile a quello di **VisRA (Visual Readability Analysis)**, che analizza feature linguistiche (come la complessità del vocabolario) per supportare l'analisi.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import spacy
import re
import string

try:
    nlp_engine = spacy.load('en_core_web_sm', disable=['parser', 'ner'])
except OSError:
    print("Modello spaCy 'en_core_web_sm' non trovato. Esegui: python -m spacy download en_core_web_sm")
    nlp_engine = None

class TextFeatureEngineer(FeatureEngineer):
    """Estende FeatureEngineer con capacità specifiche di NLP."""
    
    def engineer_text(self, text_col: str, num_keywords: int = 10) -> pd.DataFrame:
        """
        Estrae keyword principali da una colonna testuale usando TF-IDF.
        Crea colonne booleane per la presenza di ciascuna keyword top.
        """
        if nlp_engine is None or text_col not in self.df.columns: return self.df

        print(f"Inizio elaborazione testuale su '{text_col}'...")
        clean_text = self.df[text_col].astype(str).apply(self._preprocess_text)
        self.df[f'{text_col}_cleaned'] = clean_text # Salva testo pulito per usi futuri (es. WordCloud)

        # 2. TF-IDF per estrazione keyword
        tfidf = TfidfVectorizer(max_features=num_keywords, ngram_range=(1, 2), stop_words='english')
        try:
            tfidf_matrix = tfidf.fit_transform(clean_text)
            keywords = tfidf.get_feature_names_out()
            print(f" Top {num_keywords} keyword estratte: {list(keywords)}")

            # 3. Creazione colonne booleane per keyword
            for keyword in keywords:
                safe_col_name = f"cpvs_keyword_{re.sub(r'[^a-zA-Z0-9]', '_', keyword)}"
                self.df[safe_col_name] = clean_text.str.contains(keyword, regex=False).astype(int)
                
        except ValueError as e:
            print(f"Errore TF-IDF (possibile testo insufficiente): {e}")

        return self.df

    @staticmethod
    def _preprocess_text(text: str) -> str:
        """Pulisce una singola stringa (lowercase, no punctuation, lemmatization)."""
        text = text.lower().translate(str.maketrans('', '', string.punctuation + string.digits))
        doc = nlp_engine(text)
        return " ".join([t.lemma_ for t in doc if not t.is_stop and len(t.lemma_) > 2])

# --- ESECUZIONE FEATURE ENGINEERING TESTUALE ---
text_engineer = TextFeatureEngineer(processed_df)
final_df = text_engineer.engineer_text(Config.COL_CPVS, num_keywords=10)

### 8. Analisi e Gestione Distribuzioni (Visual Outlier Detection)

Prima di procedere, è essenziale comprendere la distribuzione delle variabili numeriche chiave. Le statistiche di riepilogo (come media, min, max) sono notoriamente ingannevoli e possono nascondere la vera natura dei dati, come dimostrato dall'**Anscombe's Quartet** e dal **Datasaurus Dozen**.

**Criteri di Scelta Tecnica (Istogramma + Box Plot):**
Per evitare queste trappole, usiamo una combinazione di visualizzazioni:
1.  **Istogramma:** Ci permette di vedere la *forma* della distribuzione (es. skewness, multimodalità).
2.  **Box Plot:** Utilizza il canale della **Posizione** per mostrare robusti indicatori statistici (mediana, quartili) e identificare formalmente gli *outlier* (i punti oltre i baffi).

Questa combinazione ci fornisce una visione completa e robusta, permettendoci di identificare visivamente i valori estremi che potrebbero distorcere le analisi successive.

In [ ]:
class DistributionAnalyzer(BasePlotter):
    """Visualizza le distribuzioni per identificare outlier."""
    
    def plot_distribution(self, df: pd.DataFrame, cols: list[str]):
        for col in cols:
            if col not in df.columns: continue
            
            # Setup figura doppia (Istogramma + Boxplot)
            fig, axes = plt.subplots(1, 2, figsize=(15, 6))
            fig.suptitle(f"Distribuzione di '{col}' (Pre-pulizia)", fontweight='bold')
            
            # 1. Istogramma con stima densità (KDE)
            sns.histplot(df[col].dropna(), kde=True, ax=axes[0], 
                         color=Config.COLOR_PRIMARY, alpha=0.6, edgecolor='black')
            axes[0].set_title('Istogramma e Densità')
            
            # 2. Box Plot (evidenzia outlier come punti)
            sns.boxplot(x=df[col].dropna(), ax=axes[1], 
                        color=Config.COLOR_SECONDARY, width=0.5, flierprops={'markerfacecolor':'red'})
            axes[1].set_title('Box Plot (Outlier in rosso)')
            
            sns.despine()
            plt.show()
            self._save(fig, f'02a_distribution_{col.replace(" ", "_").lower()}.png')         

# --- ESECUZIONE VISUALIZZAZIONE ---
dist_analyzer = DistributionAnalyzer()
cols_to_check = [Config.COL_DEADLINE, Config.COL_DIFF_DATES]
if Config.COL_PRICE in final_df.columns: cols_to_check.append(Config.COL_PRICE)
dist_analyzer.plot_distribution(final_df, cols_to_check)

### 8.1 Rimozione Outlier e Imputazione Finale

L'analisi visiva della cella precedente (Fase 8) ha confermato la presenza di outlier estremi. Ora agiamo su questa scoperta, seguendo il ciclo di Visual Analytics: **Analyze -> Show -> Analyze Further**.

1.  **Rimozione Outlier (Filter):** Applichiamo un filtro statistico (basato sull'Interquartile Range, IQR) per rimuovere i valori estremi che non rappresentano il "comportamento tipico" dei dati e che distorcerebbero le medie e le scale dei grafici futuri. Questo è un task di **Filtro** nel senso del mantra di Shneiderman.
2.  **Imputazione Finale:** Gestiamo i valori mancanti residui (es. in `COL_DEADLINE`) usando la *mediana* (più robusta della media agli outlier che potrebbero essere ancora presenti).

Questo ci assicura un dataset finale completo e stabile, pronto per l'analisi visiva finale.

In [ ]:
class DataRefiner:
    """Gestisce imputazione valori mancanti e rimozione outlier."""
    
    def __init__(self, df: pd.DataFrame):
        self.df = df.copy()

    def refine_all(self) -> pd.DataFrame:
        self._remove_outliers([Config.COL_DEADLINE, Config.COL_DIFF_DATES])
        self._impute_missing()
        return self.df

    def _remove_outliers(self, cols: list[str]):
        """Rimuove righe esterne a 2.5*IQR per le colonne specificate."""
        for col in cols:
            if col not in self.df.columns: continue
            
            Q1 = self.df[col].quantile(0.25)
            Q3 = self.df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 2.5 * IQR
            upper = Q3 + 2.5 * IQR
            
            initial_rows = len(self.df)
            # Manteniamo i NaN qui (saranno gestiti dall'imputer se necessario), filtriamo solo i valori validi ma estremi
            mask = (self.df[col].isna()) | ((self.df[col] >= lower) & (self.df[col] <= upper))
            self.df = self.df[mask]
            
            removed = initial_rows - len(self.df)
            if removed > 0: print(f"Rimossi {removed} outlier da '{col}' (Range accettato: [{lower:.1f}, {upper:.1f}])")

    def _impute_missing(self):
        """Riempie i valori mancanti residui con strategie standard."""
        # Esempio: riempie la scadenza mancante con la mediana (più robusta della media)
        if Config.COL_DEADLINE in self.df.columns and self.df[Config.COL_DEADLINE].isna().any():
             median_val = self.df[Config.COL_DEADLINE].median()
             self.df[Config.COL_DEADLINE].fillna(median_val, inplace=True)
             print(f"Imputati valori mancanti in '{Config.COL_DEADLINE}' con la mediana: {median_val:.0f}")

# --- ESECUZIONE REFINING ---
refiner = DataRefiner(final_df)
refined_df = refiner.refine_all()

### 8.2 Feature Engineering: Discretizzazione

L'ultima fase di preparazione dei dati è la *discretizzazione*. Trasformiamo variabili numeriche continue (come `COL_PRICE`) in variabili categoriche (es. 'Low', 'Medium', 'High').

**Motivazione Teorica (Design Space):**
Questa trasformazione è potente perché "sblocca" l'uso di canali visivi diversi e molto efficaci:
* Una variabile **Quantitativa** (Price) è mappata efficacemente a *Lunghezza* o *Posizione*.
* Trasformandola in **Nominale/Categorica**, possiamo ora mapparla a **Tonalità del Colore (Color Hue)** o usarla per raggruppare (es. in *small multiples* o *stacked bars*).

Questo arricchisce il nostro "Design Space", permettendoci di segmentare le visualizzazioni e rispondere a domande come: "Come si distribuiscono i contratti 'High' (di alto valore) tra i vari distretti?"

In [ ]:
def add_discrete_features(df: pd.DataFrame) -> pd.DataFrame:
    """Aggiunge versioni categoriche delle feature numeriche principali."""
    df = df.copy()
    
    # 1. Discretizzazione Scadenze
    for col in [Config.COL_DEADLINE, Config.COL_DIFF_DATES]:
        if col in df.columns:
            new_col = f"{col}_cat"
            try: df[new_col] = pd.qcut(df[col], 3, labels=['Short', 'Medium', 'Long'])
            except ValueError: df[new_col] = pd.cut(df[col], 3, labels=['Short', 'Medium', 'Long'])

    # 2. Discretizzazione Prezzo
    if Config.COL_PRICE in df.columns:
        new_col = f"{Config.COL_PRICE}_cat"
        try: df[new_col] = pd.qcut(df[Config.COL_PRICE], 3, labels=['Low', 'Medium', 'High'])
        except: median = df[Config.COL_PRICE].median(); df[new_col] = pd.cut(df[Config.COL_PRICE], bins=[-np.inf, median, np.inf], labels=['Low', 'High'])
            
    print("Feature discrete aggiunte (suffisso '_cat').")
    return df

refined_df = add_discrete_features(refined_df)

### 9. Analisi Testuale Avanzata (Clustering Semantico)

Portiamo l'approccio **"Analyze First"** al livello successivo. Invece di guardare solo le *keyword* (TF-IDF), vogliamo capire i "temi" semantici latenti nei testi dei contratti.

**Pipeline Analitica:**
1.  **Embedding:** Usiamo un modello (Sentence Transformer) per convertire ogni descrizione testuale (dato 1D) in un vettore numerico ad alta dimensionalità (dato n-D) che cattura il *significato semantico*.
2.  **Clustering:** Applichiamo K-Means a questi vettori per raggruppare i contratti in *cluster* tematici (es. "Lavori stradali", "Servizi IT", "Costruzione edifici").

**Valore Analitico:**
Questo processo crea una nuova, potentissima variabile **Nominale** (il `semantic_cluster`). Questo è un puro esempio di **InfoVis**: abbiamo preso dati astratti (testo) e, tramite l'analisi, abbiamo *creato* una struttura che ora possiamo visualizzare e utilizzare per segmentare l'analisi finanziaria (es. "Quanto valgono mediamente i contratti del cluster 'Lavori stradali'?").

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sentence_transformers import SentenceTransformer


class SemanticClusterer:
    """Clustering semantico con analisi finanziaria integrata."""
    
    def __init__(self, df: pd.DataFrame):
        self.df = df.copy()

    def run_clustering(self, text_col: str, n_clusters=5) -> pd.DataFrame:
        if text_col not in self.df.columns: return self.df

        print(f"Avvio Clustering Semantico su '{text_col}'...")
        try:
            model = SentenceTransformer('all-MiniLM-L6-v2')
            embeddings = model.encode(self.df[text_col].fillna("").astype(str).tolist(), 
                                    show_progress_bar=True, batch_size=128)        
            pca = PCA(n_components=2, random_state=42)
            coords = pca.fit_transform(embeddings)
            self.df['semantic_x'] = coords[:, 0]
            self.df['semantic_y'] = coords[:, 1]
            
            kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
            self.df['semantic_cluster'] = kmeans.fit_predict(embeddings)
            print(f"Clustering completato: {n_clusters} gruppi individuati.")
            
            self._analyze_financials('semantic_cluster')
            
        except Exception as e:
            print(f"Errore nel clustering (librerie mancanti?): {e}")
            
        return self.df

    def _analyze_financials(self, cluster_col: str):
        """Stampa il profilo finanziario di ogni cluster."""
        if Config.COL_PRICE not in self.df.columns: return
        
        print("\nProfilo Finanziario per Cluster Semantico:")
        stats = self.df.groupby(cluster_col)[Config.COL_PRICE].agg(
            N=('count'),
            Valore_Medio=('mean'),
            Valore_Mediano=('median'),
            Totale=('sum')
        ).sort_values(by='Valore_Medio', ascending=False)
        
        # Formattazione per output leggibile
        for col in ['Valore_Medio', 'Valore_Mediano', 'Totale']: stats[col] = stats[col].map('€{:,.0f}'.format)       
        print(stats)

# --- ESECUZIONE CLUSTERING ---
clusterer = SemanticClusterer(refined_df)
col_to_cluster = Config.COL_CPVS
clustered_df = clusterer.run_clustering(col_to_cluster, n_clusters=5)

### 10. Sommario Finale e Checkpoint di Preprocessing

Prima di passare alla Fase 2 (Visualizzazione), stampiamo un "certificato di buona salute" del dataset.

Questo riepilogo testuale funge da *checkpoint* finale, confermando le dimensioni, i nuovi tipi di dato e le statistiche di base. Ci assicura che il risultato della nostra complessa pipeline **"Analyze First"** sia un dataset robusto, arricchito e pronto per l'esplorazione visiva, che ci permetterà di generare *insight*.

In [ ]:
class DataSummarizer:
    """Genera un report testuale riassuntivo del dataset pronto."""
    
    @staticmethod
    def print_summary(df: pd.DataFrame):
        print("\n" + "="*40)
        print("DATASET MASTER: RIEPILOGO FINALE")
        print("="*40)
        print(f"Dimensioni: {df.shape[0]:,} righe, {df.shape[1]} colonne")
        print("\n--- Tipi di Dato ---")
        print(df.dtypes.value_counts())
        
        print("\n--- Statistiche Chiave (Numeriche) ---")
        # Seleziona solo alcune colonne chiave per non intasare l'output
        cols_to_summarize = [Config.COL_PRICE, Config.COL_DEADLINE, Config.COL_DIFF_DATES]
        cols_existing = [c for c in cols_to_summarize if c in df.columns]
        if cols_existing:
            print(df[cols_existing].describe().T[['mean', '50%', 'min', 'max']])

        print("\n--- Anteprima Cluster (se presenti) ---")
        if 'semantic_cluster' in df.columns:
            print(df['semantic_cluster'].value_counts().sort_index())
        
        print("="*40 + "\n")

# --- ESECUZIONE SOMMARIO E SALVATAGGIO FINALE ---
DataSummarizer.print_summary(clustered_df)

### 11. Salva Checkpoint

Salviamo il dataset "master" processato. Questo checkpoint fisico (`CLEANED_DATA`) è fondamentale per la **riproducibilità** e l'efficienza. Ci permette di ricaricare i dati puliti e arricchiti per la Fase 2 senza dover rieseguire l'intera pipeline di preprocessing (che, specialmente con l'NLP e il clustering, può essere computazionalmente costosa).

In [ ]:
clustered_df.to_csv(Config.CLEANED_DATA, index=False)
print(f"Dataset MASTER salvato in: {Config.CLEANED_DATA}")
print("Pronto per la Fase 2: Visualizzazione.")

In [ ]:
if Path(Config.CLEANED_DATA).exists():
    df_master = pd.read_csv(Config.CLEANED_DATA)
    for col in ['Signing date', 'Closing date']:
        if col in df_master.columns:
             df_master[col] = pd.to_datetime(df_master[col])
    print(f"Dataset Master caricato per la visualizzazione: {df_master.shape}")
else:
    print("ATTENZIONE: File dati puliti non trovato. Eseguire prima la Fase 1.")
    df_master = clustered_df

# FASE 2: VISUALIZATION & STORYTELLING

Con un dataset robusto e arricchito, entriamo nella fase di esplorazione visiva. L'obiettivo è applicare i principi teorici di InfoVis per "amplificare la cognizione" e generare *insight*.

## Principio Organizzativo: Mantra di Shneiderman e Keim

L'intera analisi visuale è strutturata attorno ai due mantra fondamentali della Visual Analytics:

1.  **Mantra del Visual Analytics (Keim):** Avendo già completato il passo **"Analyze first"** nella Fase 1, ora ci concentriamo su **"Show the Important"** (mostrare i pattern aggregati e i cluster), **"Zoom, filter and analyze further"** (permettere l'approfondimento) e **"Details on demand"**.

2.  **Mantra della Ricerca di Informazioni (Shneiderman):** Lo storytelling di questa fase seguirà la progressione classica:
    * **Overview First:** Inizieremo con una dashboard KPI (`DashboardBuilder`) per fornire una visione d'insieme dell'intero fenomeno.
    * **Zoom and Filter:** Utilizzeremo analizzatori specializzati (`TemporalAnalyzer`, `GeospatialAnalyzer`) per approfondire dimensioni specifiche (tempo, geografia) e filtrare i dati.
    * **Details on Demand:** Impegheremo grafici interattivi (Plotly) e tabelle che, tramite *hover* (il task *Details-on-demand*) e *lookup*, forniscono dettagli puntuali senza sovraccaricare la vista principale.

## Architettura Software

L'implementazione segue il principio di responsabilità singola. Ogni classe `Analyzer` si concentra su un tipo di dati o un task analitico, ereditando da `BasePlotter` per garantire coerenza nel **Design Space**.

## Note Metodologiche Visuali

* **Scala Logaritmica:** Utilizzata estensivamente per i dati finanziari (prezzo). Come visto nella Fase 1, questi dati sono *right-skewed*. La scala logaritmica è una trasformazione che ci permette di visualizzare e confrontare ordini di grandezza diversi (da €1k a €100M) nello stesso grafico, senza che i valori più piccoli vengano schiacciati a zero.
* **Mappe Coropletiche:** Tecnica classica per **Dati 2D-Map**. Si tratta di un substrato geografico (che ricade nella **SciVis**) su cui mappiamo dati astratti (un task di **InfoVis**), come il prezzo medio o il volume, usando il canale del **Colore (Saturazione)**.
* **KDE (Kernel Density Estimation):** Usato per l'analisi di densità bivariata. È superiore a uno scatterplot standard in presenza di *overplotting* (migliaia di punti sovrapposti), poiché stima e visualizza la densità (spesso tramite *colore*) permettendo di identificare i *cluster*.

## 1. Panoramica Esecutiva (KPI Dashboard)

Iniziamo la nostra analisi seguendo la prima regola del mantra di Shneiderman: **"Overview first"**.

Questa dashboard composita funge da "cruscotto" e fornisce il contesto generale. Combina indicatori chiave di performance (KPI) numerici (per un *lookup* immediato dei valori) con grafici di alto livello (per la percezione dei *pattern*). L'obiettivo è dare all'utente un orientamento immediato sulle dimensioni del fenomeno:
* **Volume e Valore Totale:** Quanto è grande il dataset?
* **Durata Media:** Quanto tempo richiedono i progetti?
* **Top Player:** Quali distretti muovono più denaro?
* **Trend Generale:** Il mercato è in crescita o contrazione?

In [ ]:
import matplotlib.gridspec as gridspec

class DashboardBuilder(BasePlotter):
    """Costruisce dashboard riepilogative complesse."""
    
    def build_main_kpi(self, df: pd.DataFrame):
        fig = plt.figure(figsize=(18, 12))
        gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.4, wspace=0.3)
        fig.patch.set_facecolor('#F8F9FA') # Sfondo leggero professionale

        # --- RIGA 1: KPI CARDS ---
        kpi1 = fig.add_subplot(gs[0, 0])
        self._draw_kpi_card(kpi1, f"{len(df):,}", "Contratti Totali", "Dataset analizzato", Config.COLOR_PRIMARY)
        
        kpi2 = fig.add_subplot(gs[0, 1])
        avg_val = df[Config.COL_PRICE].mean()
        self._draw_kpi_card(kpi2, f"€{avg_val/1e6:.1f}M", "Valore Medio", "Per contratto", "#2ECC71")
        
        kpi3 = fig.add_subplot(gs[0, 2])
        avg_days = df[Config.COL_DEADLINE].mean()
        self._draw_kpi_card(kpi3, f"{avg_days:.0f}", "Giorni Medi", "Durata esecuzione", Config.COLOR_SECONDARY)

        # --- RIGA 2: ANALISI DISTRETTI & PREZZI ---
        # Top 5 Distretti per Valore
        ax_dist = fig.add_subplot(gs[1, :2])
        top_5 = df.groupby(Config.COL_DISTRICT)[Config.COL_PRICE].sum().nlargest(5).sort_values(ascending=True)
        bars = ax_dist.barh(top_5.index, top_5.values, color=sns.color_palette("viridis", 5))
        ax_dist.set_title("Top 5 Distretti per Valore Totale Contratti", fontweight='bold')
        self._format_currency(ax_dist, 'x', 'M')
        sns.despine(ax=ax_dist, left=True)

        # Distribuzione Prezzi (Boxen Plot per gestire code lunghe)
        ax_price = fig.add_subplot(gs[1, 2])
        if Config.COL_AWARD in df.columns:
            sns.boxenplot(data=df, x=Config.COL_AWARD, y=Config.COL_PRICE, ax=ax_price, palette="Set2")
            ax_price.set_yscale('log')
            ax_price.set_title("Distribuzione Prezzi per Criterio", fontweight='bold')
            ax_price.set_ylabel("Prezzo (€, scala log)")
            ax_price.set_xlabel("")
            plt.setp(ax_price.get_xticklabels(), rotation=15, ha="right")

        # --- RIGA 3: TREND TEMPORALE (Doppio Asse) ---
        ax_trend = fig.add_subplot(gs[2, :])
        yearly = df.groupby(Config.COL_YEAR).agg(
            Count=(Config.COL_YEAR, 'size'), 
            Value=(Config.COL_PRICE, 'sum')
        )
        
        # Linea Volume (sx)
        ax_trend.plot(yearly.index, yearly['Count'], marker='o', color=Config.COLOR_PRIMARY, lw=3, label='N. Contratti')
        ax_trend.set_ylabel('Numero Contratti', color=Config.COLOR_PRIMARY, fontweight='bold')
        ax_trend.tick_params(axis='y', labelcolor=Config.COLOR_PRIMARY)
        
        # Barre Valore (dx)
        ax_val = ax_trend.twinx()
        ax_val.bar(yearly.index, yearly['Value'], color=Config.COLOR_SECONDARY, alpha=0.5, label='Valore Totale')
        ax_val.set_ylabel('Valore Totale (€)', color=Config.COLOR_SECONDARY, fontweight='bold')
        ax_val.tick_params(axis='y', labelcolor=Config.COLOR_SECONDARY)
        self._format_currency(ax_val, 'y', 'M')
        
        ax_trend.set_title("Trend Temporale: Volume vs Valore", fontweight='bold')
        
        # Legenda unica manuale
        lines, labels = ax_trend.get_legend_handles_labels()
        lines2, labels2 = ax_val.get_legend_handles_labels()
        ax_trend.legend(lines + lines2, labels + labels2, loc='upper left')

        plt.suptitle('Dashboard Analitica - Appalti Pubblici Portogallo', fontsize=22, fontweight='bold', y=0.95)
        plt.show()
        self._save(fig, '10_kpi_dashboard_executive.png')

    def _draw_kpi_card(self, ax, value, title, subtitle, color):
        """Helper interno per disegnare una 'card' KPI pulita."""
        ax.axis('off')
        # Rettangolo di sfondo con bordo colorato
        rect = plt.Rectangle((0.05, 0.05), 0.9, 0.9, transform=ax.transAxes, 
                             fc='white', ec=color, lw=2, alpha=1, zorder=1)
        ax.add_patch(rect)
        
        ax.text(0.5, 0.6, value, transform=ax.transAxes, ha='center', va='center', 
                fontsize=32, fontweight='bold', color=color, zorder=2)
        ax.text(0.5, 0.35, title, transform=ax.transAxes, ha='center', va='center', 
                fontsize=14, fontweight='bold', color='#2C3E50', zorder=2)
        ax.text(0.5, 0.2, subtitle, transform=ax.transAxes, ha='center', va='center', 
                fontsize=10, color='#95A5A6', zorder=2)

# --- ESECUZIONE DASHBOARD ---
dashboard = DashboardBuilder()

### 1.1 Interpretazione Dashboard KPI

**Scelta Tecnica: Dashboard Multi-View**

Combina 4 visualizzazioni complementari per fornire overview completa del dataset.

**Design Space:**
* **KPI Cards:** Ridondanza testuale per sintesi immediata (memorabilità)
* **Bar Chart (Top 5):** Lunghezza su asse comune = canale più efficace per confronti quantitativi
* **Boxen Plot:** Evoluzione del box plot, mostra più quantili → migliore per distribuzioni complesse
* **Dual-Axis Chart:** Due scale Y condivise su asse X temporale → confronto trend con unità incomparabili

**Layout:** GridSpec per gerarchia visiva e confronto parallelo

---

**Interpretazione**

**Concentrazione Geografica Estrema**
- Top 3 distretti (Beja, Porto, Lisboa) = maggioranza del valore totale
- Forte polarizzazione: opportunità economiche concentrate territorialmente

**Eterogeneità Criteri**
- Distribuzione prezzi (scala log) mostra:
  - Ampia variabilità intra-criterio
  - Numerosi outlier estremi (>10^7€)
  - Mediane differenti tra criteri = segmenti di mercato distinti

**Decoupling Volume-Valore**
- **2014:** Picco valore (€700M) con volume moderato → pochi mega-appalti
- **2020:** Picco volume (750+ contratti) ma valore sotto media → frammentazione in appalti piccoli
- **2018-2019:** Ripresa non proporzionale (volume cresce più del valore)
- **2021-2022:** Crollo simultaneo → shock sistemico

**Conclusione:** Mercato asimmetrico e ciclico. Assenza di elasticità volume-valore: alcuni anni trainati da mega-progetti, altri da frammentazione operativa. Distribuzione altamente variabile senza comportamento stabile.

In [ ]:
dashboard.build_main_kpi(df_master)

## 2. Analisi Temporale e Stagionalità

Dopo l'Overview, passiamo alla fase di **"Zoom and Filter"**. In questa sezione, "zoomiamo" sulla dimensione temporale per analizzare **dati temporali (Time-Series)**. L'obiettivo è scoprire pattern ciclici (stagionalità) e trend evolutivi.

In [ ]:
class TemporalAnalyzer(BasePlotter):

    def plot_quarter_distribution(self, df: pd.DataFrame):
        """
        Grafico della distribuzione trimestrale dei contratti (Q1–Q4)
        utilizzando Small Multiples (Faceting) per mostrare i trend individuali.
        """

        if 'Signing Month' not in df.columns:
            print("Colonna 'Signing Month' non trovata")
            return
    
        # --- Calcolo del Quarter ---
        df = df.copy()
        if Config.COL_DATE_SIGN in df.columns and not df[Config.COL_DATE_SIGN].isnull().all():
             # Metodo più robusto se la data completa è disponibile
            df['Quarter'] = pd.to_datetime(df[Config.COL_DATE_SIGN]).dt.quarter
        else:
             # Fallback sul mese
            df['Quarter'] = ((df['Signing Month'] - 1) // 3) + 1
            
        df['Quarter'] = df['Quarter'].fillna(0).astype(int)

        # --- Aggregazione per Anno × Quarter ---
        quarter_data = df.groupby([Config.COL_YEAR, 'Quarter']).size().unstack(fill_value=0)

        # Assicuriamoci che tutti e 4 i trimestri esistano, anche se vuoti
        for q in [1, 2, 3, 4]:
            if q not in quarter_data.columns:
                quarter_data[q] = 0

        # --- Plot: Small Multiples (Griglia 2x2) ---
        # sharey=True è FONDAMENTALE per un confronto onesto
        fig, axes = plt.subplots(2, 2, figsize=(16, 10), sharey=True, sharex=True)
        
        # Appiattiamo l'array 2x2 di assi per un loop più semplice
        axes_list = [axes[0, 0], axes[0, 1], axes[1, 0], axes[1, 1]]
        
        colors_quarter = {
            1: "#3498DB",  # blu Q1
            2: "#E67E22",  # arancione Q2
            3: "#2ECC71",  # verde Q3
            4: "#E74C3C"   # rosso Q4 (lo rendiamo più "caldo" del viola)
        }

        for i, q in enumerate([1, 2, 3, 4]):
            ax = axes_list[i]
            data_to_plot = quarter_data[q]
            
            ax.plot(
                data_to_plot.index, 
                data_to_plot.values,
                label=f"Q{q}",
                marker='o',
                linewidth=2.5,
                markersize=7,
                color=colors_quarter.get(q)
            )
            
            # --- Stile per ogni subplot ---
            ax.set_title(f"Andamento Trimestre {q}", fontsize=14, fontweight="bold")
            ax.grid(axis='y', linestyle='--', alpha=0.7)
            ax.tick_params(axis='x', rotation=45)
            sns.despine(ax=ax)

        # --- Stile Globale ---
        fig.suptitle(
            "Andamento Contratti per Trimestre (Small Multiples)",
            fontsize=18, fontweight="bold", color="#2C3E50", y=1.03
        )
        
        # Etichette comuni per assi condivisi
        fig.supxlabel("Anno", fontsize=14, fontweight="bold", y=0.02)
        fig.supylabel("N. Contratti", fontsize=14, fontweight="bold", x=0.06)

        plt.tight_layout()
        plt.show()

        # --- Salvataggio ---
        self._save(fig, "11_a_quarter_distribution_small_multiples.png")

    def plot_seasonality_heatmap(self, df: pd.DataFrame):
        """Genera heatmap stagionale Mese × Anno."""
        if 'Signing Month' not in df.columns: 
            print("Colonna 'Signing Month' non trovata")
            return
        
        heatmap_data = df.pivot_table(
            index='Signing Month', columns=Config.COL_YEAR, 
            values=Config.COL_ID if Config.COL_ID in df.columns else df.columns[0], 
            aggfunc='count'
        ).fillna(0)
        
        fig, ax = plt.subplots(figsize=(14, 8))
        # Heatmap con palette professionale blu-bianco-rosso
        sns.heatmap(heatmap_data, cmap='RdYlBu_r', annot=True, fmt='.0f', 
                   linewidths=1, linecolor='white', ax=ax, 
                   cbar_kws={
                       'label': 'N. Contratti',
                       'shrink': 0.8,
                       'pad': 0.02,
                       'aspect': 30
                   }, 
                   vmin=0, annot_kws={'fontsize': 10, 'weight': 'bold'})
        
        ax.set_title("Heatmap Stagionale: Intensità Contratti per Mese/Anno", 
                    fontweight='bold', fontsize=16, pad=20, color='#2C3E50')
        ax.set_xlabel("Anno", fontweight='bold', fontsize=12)
        ax.set_ylabel("Mese", fontweight='bold', fontsize=12)
        ax.set_yticklabels(['Gen', 'Feb', 'Mar', 'Apr', 'Mag', 'Giu', 
                            'Lug', 'Ago', 'Set', 'Ott', 'Nov', 'Dic'], rotation=0)
        
        # Colorbar a destra più visibile
        cbar = ax.collections[0].colorbar
        cbar.ax.tick_params(labelsize=11)
        cbar.set_label('N. Contratti', fontsize=12, weight='bold')
        
        plt.tight_layout()
        plt.show()
        self._save(fig, '11_b_seasonality_heatmap.png')
    
    def plot_criteria_evolution(self, df: pd.DataFrame):
        """Genera stacked area chart evoluzione criteri."""
        if Config.COL_AWARD not in df.columns: 
            print("Colonna criterio aggiudicazione non trovata")
            return
        
        pivot = df.pivot_table(
            index=Config.COL_YEAR, columns=Config.COL_AWARD, 
            values=Config.COL_PRICE, aggfunc='sum'
        ).fillna(0)
        
        fig, ax = plt.subplots(figsize=(14, 7))
        colors_criteria = ['#3498DB', '#E74C3C', '#2ECC71', '#F39C12', '#9B59B6']
        ax.stackplot(pivot.index, *pivot.T.values, 
                     labels=pivot.columns, alpha=0.75, colors=colors_criteria,
                     edgecolor='white', linewidth=1.5)
        
        ax.set_title("Evoluzione Criteri di Aggiudicazione (Valore Cumulativo)", 
                    fontweight='bold', fontsize=16, pad=15, color='#2C3E50')
        ax.set_xlabel("Anno", fontweight='bold', fontsize=12)
        ax.set_ylabel("Valore Totale (€)", fontweight='bold', fontsize=12)
        self._format_currency(ax, 'y', 'M')
        ax.legend(loc='upper left', framealpha=0.95, fontsize=10)
        ax.grid(axis='y', alpha=0.3, linestyle='--')
        plt.tight_layout()
        plt.show()
        self._save(fig, '12_criteria_evolution_stacked.png')

# --- INIZIALIZZAZIONE ANALYZER ---
temp_analyzer = TemporalAnalyzer()

### 2.1 Grafico 1: Heatmap Stagionale Mese × Anno

**Scelta Tecnica: Heatmap**

Mappa una griglia 2D (Mese × Anno) su una variabile quantitativa (N. Contratti) usando il canale del **Colore (Saturazione)**.

**Design Space:**
* **Asse X:** Anno (Quantitativo ordinale)
* **Asse Y:** Mese (Categorico ordinale)
* **Colore:** Intensità (Saturazione) per N. Contratti (Quantitativo)

Sfrutta la percezione preattentiva del colore per identificare:
* **Pattern verticali:** Trend annuali
* **Pattern orizzontali:** Stagionalità mensile
* **Anomalie:** Celle anomale rispetto al contesto

---

**Interpretazione**

**Assenza di Stagionalità Ricorrente**
- Nessun pattern mensile stabile lungo la serie storica
- Variabilità anno-per-anno predominante rispetto alle differenze intra-anno
- Nessun mese presenta comportamento costantemente alto o basso

**Dominanza dell'Effetto Anno**
- Picchi concentrati nel periodo 2017-2020 (rosso intenso)
- Mesi estivi (Giu-Ago) molto attivi solo negli anni ad alto volume complessivo
- Nessuna "valle estiva" sistematica: estate 2017-2019 tra i periodi più intensi

**Pattern Q4 Debole**
- Ottobre mostra intensità elevata (2017-2020), ma non costante
- Novembre-Dicembre variabili, senza pattern di fine anno stabile

**Conclusione:** I volumi mensili sono guidati da cicli annuali e dinamiche di periodo, non da stagionalità ricorrente. Il comportamento è determinato dall'andamento generale del mercato più che da fattori stagionali intrinseci.

In [ ]:
# --- ESECUZIONE HEATMAP STAGIONALE ---
temp_analyzer.plot_seasonality_heatmap(df_master)

### 2.2 Grafico 2: Andamento Trimestrale (Small Multiples)

**Scelta Tecnica: Small Multiples (Faceting)**

Risolve il problema degli "spaghetti plot" separando i 4 trimestri in grafici distinti affiancati.

**Design Space:**
* **Asse X:** Anno (Quantitativo)
* **Asse Y:** N. Contratti (Quantitativo) → scala condivisa tra facet
* **Facet:** Trimestre (Categorico) → 4 pannelli separati
* **Linea:** Segno che collega punti temporali consecutivi

La scala Y condivisa (`sharey=True`) permette confronto diretto delle magnitudini tra trimestri.

---

**Interpretazione**

**Sincronizzazione Totale: Conferma Effetto-Anno**
- Tutti e 4 i trimestri seguono lo stesso pattern identico
- Crescita parallela 2010-2019, picco 2017-2019, crollo 2020-2022
- Forme delle curve sovrapponibili tra trimestri

**Assenza di Dominanza Stagionale**
- Nessun trimestre sistematicamente superiore agli altri
- Picchi di magnitudo simile (~175-230 contratti) per tutti i trimestri
- Differenze tra trimestri < variabilità anno-per-anno

**Q3 (Estate) Non È Valle**
- Trimestre 3 (Lug-Set) tra i più forti nel periodo di picco
- Valori 2017-2020 comparabili o superiori a Q4
- Conferma assenza di stagionalità estiva negativa

**Crollo Uniforme Post-2020**
- Tutti i trimestri collassano simultaneamente verso zero nel 2022
- Nessun trimestre "resistente": shock sistemico non stagionale

**Conclusione:** I trimestri sono perfetti duplicati dello stesso ciclo pluriennale. Il mercato è guidato da dinamiche macroeconomiche/normative, non da fattori stagionali intrinseci.

In [ ]:
temp_analyzer.plot_quarter_distribution(df_master)

### 2.3 Grafico 3: Evoluzione Criteri di Aggiudicazione (Stacked Area)

**Scelta Tecnica: Stacked Area Chart**

Analizza la composizione temporale del mercato attraverso criteri di aggiudicazione.

**Design Space:**
* **Asse X:** Anno (Quantitativo)
* **Asse Y:** Valore € (Quantitativo) → Posizione verticale
* **Aree:** Volume monetario per criterio
* **Colore:** Tonalità (Hue) per distinguere categorie nominali

Visualizza simultaneamente:
1. **Trend totale:** Altezza complessiva = andamento del mercato
2. **Composizione:** Dimensione relativa delle aree = quota per criterio

---

**Interpretazione**

**Fase 1 (2010-2014): Dominio del Criterio 0**
- Crescita esplosiva trainata dal criterio 0 (blu)
- Picco: €640M nel 2014
- Criteri 1 e 2 marginali (<5%)

**Fase 2 (2014-2015): Discontinuità**
- Crollo del criterio 0: da €640M a €50M (-92%)
- Sostituzione immediata con criteri 1 (arancione) e 2 (verde)
- Valore totale dimezzato (€640M → €300M)
- Probabile intervento legislativo che ha eliminato il criterio 0

**Fase 3 (2016-2022): Declino Strutturale**
- Contrazione progressiva di tutti i criteri
- Valore totale → 0 nel 2022
- Possibile fine dei bandi o migrazione verso altre modalità

**Conclusione:** Punto di svolta netto nel 2014-2015 seguito da collasso irreversibile. Non evoluzione graduale, ma shock normativo con declino sistemico del settore.

In [ ]:
temp_analyzer.plot_criteria_evolution(df_master)

## 3. Analisi Geospaziale (Zoom sulla Dimensione Territoriale)

Continuiamo la nostra fase di **"Zoom and Filter"**, spostando il focus dal *tempo* allo *spazio*. L'obiettivo è capire *dove* vengono allocati i fondi pubblici.

Utilizzeremo mappe coropletiche, che sono la tecnica di visualizzazione standard per **Dati 2D-Map** o **Dati Geografici**. Come evidenziato da esempi storici (la mappa del colera di John Snow, 1854), la visualizzazione spaziale è fondamentale per generare *insight* legati alla localizzazione.

In [ ]:
import json

class GeospatialAnalyzer(BasePlotter):
    """Analisi geografica."""

    def __init__(self):
        super().__init__()
        self.geojson = None
        self.district_mapping = {
            'Região Autónoma dos Açores': 'Açores',
            'Região Autónoma da Madeira': 'Madeira'
        }
        
        geojson_path = Config.GEOJSON.parent / 'georef-portugal-distrito.geojson'
        if geojson_path.exists():
            with open(geojson_path, 'r', encoding='utf-8') as f:
                self.geojson = json.load(f)

    def plot_choropleth(self, df: pd.DataFrame, metric_col: str, agg_func: str, title: str, filename: str):
        """Genera mappa coropletica."""
        if self.geojson is None: 
            return
        
        df_mapped = df.copy()
        if Config.COL_DISTRICT in df_mapped.columns:
            df_mapped[Config.COL_DISTRICT] = df_mapped[Config.COL_DISTRICT].replace(self.district_mapping)
        
        dist_data = df_mapped.groupby(Config.COL_DISTRICT)[metric_col].agg(agg_func).reset_index()
        dist_data.columns = [Config.COL_DISTRICT, 'Value']
        
        # Calcola centroidi per le etichette
        centroids = []
        for feat in self.geojson['features']:
            name = feat['properties']['dis_name']
            if name in dist_data[Config.COL_DISTRICT].values:
                geom = feat['geometry']['coordinates']
                if feat['geometry']['type'] == 'MultiPolygon':
                    coords = [pt for poly in geom for ring in poly for pt in ring]
                else:
                    coords = [pt for ring in geom for pt in ring]
                lons = [c[0] for c in coords]
                lats = [c[1] for c in coords]
                centroids.append({
                    'District': name,
                    'lon': sum(lons) / len(lons),
                    'lat': sum(lats) / len(lats)
                })
        
        fig = px.choropleth_mapbox(
            dist_data, geojson=self.geojson, locations=Config.COL_DISTRICT,
            featureidkey='properties.dis_name', color='Value',
            color_continuous_scale='Teal', mapbox_style="carto-positron",
            zoom=5.5, center={"lat": 39.5, "lon": -8.0}, opacity=0.75,
            hover_name=Config.COL_DISTRICT, hover_data={'Value': ':,.0f'}
        )
        
        # Aggiungi etichette distretto
        for c in centroids:
            fig.add_scattermapbox(
                lon=[c['lon']], lat=[c['lat']], mode='text',
                text=[c['District']], textfont=dict(size=10, color='black'),
                hoverinfo='skip', showlegend=False
            )
        
        fig.update_layout(title_text=title, margin={"r":0,"t":50,"l":0,"b":0}, height=600)
        
        html_path = Config.PLOTS_DIR / filename
        fig.write_html(html_path)
        fig.show()

    def plot_top_districts_bars(self, df: pd.DataFrame):
        """Top 15 distretti per Totale e Media."""
        metrics = df.groupby(Config.COL_DISTRICT)[Config.COL_PRICE].agg(
            Total='sum', Average='mean'
        ).reset_index()

        top15_total = metrics.sort_values('Total', ascending=True).tail(15)
        top10_total = metrics.sort_values('Total', ascending=False).head(15)
        
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7), gridspec_kw={'width_ratios': [2, 1]})
        
        norm = plt.Normalize(vmin=top15_total['Total'].min(), vmax=top15_total['Total'].max())
        colors_total = plt.cm.Reds(norm(top15_total['Total']))
        ax1.barh(top15_total[Config.COL_DISTRICT], top15_total['Total']/1e6, color=colors_total, edgecolor='#2C3E50')
        ax1.set_title("Top 15 Distretti per Valore Totale", fontsize=14, weight='bold')
        ax1.set_xlabel("Valore Totale (€M)", weight='bold')
        
        sm = plt.cm.ScalarMappable(cmap='Reds', norm=norm)
        sm.set_array([])
        plt.colorbar(sm, ax=ax1, label='Intensità Valore (€)').ax.yaxis.label.set_weight('bold')
        
        wedges, texts, autotexts = ax2.pie(
            top10_total['Total'], labels=top10_total[Config.COL_DISTRICT],
            autopct='%1.1f%%', startangle=90, counterclock=False,
            colors=plt.cm.Reds(np.linspace(0.8, 0.2, 15)),
            wedgeprops={'edgecolor': 'white', 'linewidth': 1}
        )
        ax2.set_title("Top 5 Distretti\n(Quota sul Totale dei Top 15)", weight='bold')
        plt.setp(autotexts, size=9, weight='bold', color='white')
        import matplotlib.patheffects as path_effects
        for text in autotexts:
            text.set_path_effects([path_effects.withStroke(linewidth=2, foreground='black')])

        plt.tight_layout()
        plt.show()
        self._save(fig, '15a_top15_districts_total.png')

        top15_avg = metrics.sort_values('Average', ascending=True).tail(15)
        top10_avg = metrics.sort_values('Average', ascending=False).head(15)
        
        fig, (ax3, ax4) = plt.subplots(1, 2, figsize=(18, 7), gridspec_kw={'width_ratios': [2, 1]})
        
        norm_avg = plt.Normalize(vmin=top15_avg['Average'].min(), vmax=top15_avg['Average'].max())
        colors_avg = plt.cm.Blues(norm_avg(top15_avg['Average']))
        ax3.barh(top15_avg[Config.COL_DISTRICT], top15_avg['Average']/1e6, color=colors_avg, edgecolor='#2C3E50')
        ax3.set_title("Top 15 Distretti per Valore Medio", fontsize=14, weight='bold')
        ax3.set_xlabel("Valore Medio (€M)", weight='bold')
        
        sm_avg = plt.cm.ScalarMappable(cmap='Blues', norm=norm_avg)
        sm_avg.set_array([])
        plt.colorbar(sm_avg, ax=ax3, label='Intensità Valore (€)').ax.yaxis.label.set_weight('bold')
        
        wedges, texts, autotexts = ax4.pie(
            top10_avg['Average'], labels=top10_avg[Config.COL_DISTRICT],
            autopct='%1.1f%%', startangle=90, counterclock=False,
            colors=plt.cm.Blues(np.linspace(0.8, 0.2, 10)),
            wedgeprops={'edgecolor': 'white', 'linewidth': 1}
        )
        ax4.set_title("Top 15 Distretti\n(Confronto Medie)", weight='bold')
        plt.setp(autotexts, size=9, weight='bold', color='white')
        for text in autotexts:
            text.set_path_effects([path_effects.withStroke(linewidth=2, foreground='black')])

        plt.tight_layout()
        plt.show()
        self._save(fig, '15b_top15_districts_avg.png')

    def plot_scatter_volume_value(self, df: pd.DataFrame):
        """Genera scatter plot volume vs valore."""
        metrics = df.groupby(Config.COL_DISTRICT).agg(
            Total_Value=(Config.COL_PRICE, 'sum'),
            Average_Value=(Config.COL_PRICE, 'mean'),
            Contracts=(Config.COL_PRICE, 'count')
        ).reset_index()
        
        fig = px.scatter(
            metrics, x='Contracts', y='Total_Value', size='Average_Value',
            color=Config.COL_DISTRICT, hover_name=Config.COL_DISTRICT,
            size_max=50, title="Relazione Volume-Valore per Distretto",
            labels={'Contracts': 'Numero Contratti', 'Total_Value': 'Valore Totale (€)', 'Average_Value': 'Valore Medio (€)'},
            hover_data={'Contracts': True, 'Total_Value': ':,.0f', 'Average_Value': ':,.0f'}
        )
        fig.update_layout(height=700, plot_bgcolor='rgba(248,249,250,1)', font=dict(family="Arial", size=12))
        fig.write_html(Config.PLOTS_DIR / '16_scatter_districts_vol_val.html')
        fig.show()

geo_analyzer = GeospatialAnalyzer()

### 3.1 Grafico 1: Mappa Coropletica Valore Medio per Distretto

**Scelta Tecnica: Choropleth Map (InfoVis su Substrato SciVis)**

Mappa dati astratti (valore medio contratti) su geometria geografica intrinseca.

**Design Space:**
* **Substrato:** Geografia del Portogallo (SciVis)
* **Variabile:** Valore medio contratti (Quantitativo)
* **Canale:** Saturazione colore (intensità = valore maggiore)

**Metrica:** Valore medio normalizza per dimensione distretto, rivela tipologia contratti (mega-progetti vs frammentazione) indipendentemente da volume totale o popolazione.

---

**Interpretazione**

**Anomalia Beja: Dominanza Assoluta**
- Valore medio ~€4M, 4-8x superiore a tutti gli altri distretti
- Non è area metropolitana → segnale di pochi mega-progetti concentrati
- Possibile hub infrastrutturale (energia, trasporti, opere idrauliche)

**Aree Metropolitane: Valori Medi Nella Norma**
- Lisboa e Porto con valori medio-bassi (~€0.5-1M)
- Mix eterogeneo: molti appalti piccoli/medi diluiscono la media
- Mercato frammentato e diversificato

**Pattern Centro-Nord**
- Braga, Vila Real, Viseu: valori medi superiori a capitali
- Distretti interni con basso volume totale ma alto valore medio
- Concentrazione su opere infrastrutturali strategiche

**Conclusione:** La geografia del valore medio è inversa rispetto al volume. Beja emerge come outlier estremo, mentre le metropoli mostrano frammentazione. Il valore medio identifica hub di mega-progetti, non necessariamente centri economici tradizionali.

In [ ]:
# --- ESECUZIONE: Mappa Valore Medio ---
geo_analyzer.plot_choropleth(
    df_master, 
    Config.COL_PRICE, 
    'mean', 
    "Valore Medio Contratti per Distretto (€)", 
    "13_map_mean_value.html"
)

### 3.2 Grafico 2: Mappa Coropletica Volume Contratti per Distretto

**Scelta Tecnica: Choropleth per Vista Complementare**

Seconda mappa geografica per analisi multidimensionale attraverso **viste multiple coordinate** (Keim, 2002). Il confronto mentale tra mappe implementa il concetto di **Brushing and Linking** manuale: identificare pattern in una vista (G3.1: "Beja ha valore medio alto") e verificarlo nell'altra (G3.2: "Beja ha volume basso") per generare insight combinato attraverso cognizione.

**Design Space (Munzner):**
* **Substrato:** Geografia 2D (SciVis)
* **Segni:** Aree (poligoni distrettuali)
* **Canale:** Saturazione colore per variabile quantitativa (N. Contratti)

**Metrica:** Volume assoluto rivela densità operativa e maturità del mercato locale.

---

**Interpretazione**

**Inversione Geografica Rispetto a Valore Medio**
- **Lisboa e Porto:** Dominanza assoluta (~600+ e ~500 contratti)
- **Beja:** Volume bassissimo (~150) ma valore medio €4M
- Pattern inverso conferma mercati strutturalmente diversi

**Tipologie di Mercato (Insight da Brushing Mentale)**

*Mercato Metropolitano Frammentato (Lisboa, Porto):*
- Alto volume + valore medio basso/medio
- Appalti operativi, manutenzioni, servizi
- Mercato maturo e diversificato

*Mercato Mega-Progetti (Beja, sud):*
- Basso volume + valore medio altissimo  
- Pochi contratti strategici ad alto importo
- Specializzazione infrastrutturale (energia, idraulica, trasporti)

*Deserto Amministrativo (Interni nord-est):*
- Basso volume + basso valore medio
- Assenza opere strategiche e frammentazione limitata

**Conclusione:** Le viste combinate (principio **"Overview first, Zoom and Filter, Details-on-demand"** - Shneiderman) rivelano geografia duale: metropoli con mercati densi vs periferie con investimenti concentrati. Beja emerge come outlier di concentrazione infrastrutturale.

In [ ]:
# --- ESECUZIONE: Mappa Volume ---
geo_analyzer.plot_choropleth(
    df_master, 
    Config.COL_PRICE, 
    'count', 
    "Volume Contratti per Distretto (N. Contratti)", 
    "14_map_volume.html"
)

### 3.3 Grafici 3-4: Top 15 Distretti per Valore Totale e Medio

**Scelta Tecnica: Bar Chart Orizzontale + Pie Chart**

Due visualizzazioni parallele per confrontare ranking assoluto vs normalizzato.

**Design Space (Bar Chart):**
* **Segni:** Barre orizzontali
* **Canale primario:** Lunghezza → Valore € (Quantitativo)
* **Canale ridondante:** Saturazione colore → stesso valore (Data Redundancy per migliore recall, Borkin et al.)
* **Orientamento:** Orizzontale per leggibilità etichette nominali lunghe

**Design Space (Pie Chart):**
* **Canale:** Angolo/Area (canali deboli per quantitativo)
* **Uso:** Solo overview compositiva Top 5, supporto al bar chart

---

**Interpretazione**

**Inversione Totale del Ranking**

*Valore Totale (Grafico sinistra):*
- **Beja domina:** €650M (~17% del totale Top 15)
- Top 3 (Beja, Porto, Lisboa): 42% del valore
- Segue distribuzione demografica tradizionale

*Valore Medio (Grafico destra):*
- **Beja outlier estremo:** €4M/contratto (3x superiore al secondo)
- **Madeira e isole salgono:** Regioni periferiche scalano ranking
- **Metropoli crollano:** Lisboa e Porto fuori Top 3

**Conferma Insight Mappe**
- Alto valore totale ≠ Alto valore medio
- Beja: pochi mega-progetti concentrati (€650M / ~150 contratti = €4.3M)
- Lisboa: frammentazione operativa (€500M / ~600 contratti = €0.8M)

**Pie Chart: Concentrazione Moderata**
- Top 5 = 55% del totale (distribuzione meno concentrata di quanto sembri)
- Long tail significativa (10 distretti = 45%)

**Conclusione:** I due ranking rivelano economie duali. Il valore totale segue popolazione/PIL, il valore medio identifica hub infrastrutturali strategici indipendenti da dimensione demografica.

In [ ]:
geo_analyzer.plot_top_districts_bars(df_master)

### 3.4 Grafico 5: Scatter Plot Multivariato Volume-Valore-Medio

**Scelta Tecnica: Bubble Chart (4 Dimensioni su 2D)**

Sintetizza 4 variabili in unico spazio visivo per identificare cluster e outlier.

**Design Space:**
* **Asse X:** N. Contratti (Quantitativo) → Posizione
* **Asse Y:** Valore Totale € (Quantitativo) → Posizione
* **Area Bolla:** Valore Medio € (Quantitativo) → Dimensione
* **Colore:** Distretto (Nominale) → Tonalità

Usa **3 canali quantitativi gerarchici**: Posizione (massima precisione) per confronti primari, Area (meno precisa) per variabile terziaria.

---

**Interpretazione**

**Quattro Archetipi Identificati**

*1. Outlier Mega-Progetti (Beja):*
- Posizione: (100, €600M)
- Bolla gigante (€4M medio)
- Isolato: alto valore totale con bassissimo volume

*2. Metropoli Frammentate (Lisboa, Porto):*
- Posizione: cluster destro (500-700 contratti)
- Alto valore totale (€450-500M)
- Bolle piccole (€0.8-1M medio)

*3. Hub Regionali (Aveiro, Braga):*
- Posizione: centro-destra (250-350 contratti)
- Valore medio (€200-250M)
- Bolle medie (€1-1.5M)

*4. Periferie (cluster basso-sinistra):*
- <150 contratti, <€150M
- Bolle variabili ma piccole in valore assoluto

**Correlazione Volume-Valore Non Lineare**
- Relazione generale positiva ma debole
- Beja rompe completamente il pattern
- Due "sentieri" distinti: frammentazione vs concentrazione

**Conclusione:** Lo scatter multivariato conferma dualità strutturale. Posizione spaziale rivela volume/valore assoluto, dimensione bolla espone strategia (mega-progetti vs operativo). Beja è anomalia statisticamente e visivamente isolata.

In [ ]:
geo_analyzer.plot_scatter_volume_value(df_master)

## 4. Approfondimenti Finanziari (Zoom sulla Distribuzione)

In questa sezione, applichiamo uno **"Zoom"** sulla variabile più importante del dataset: il Prezzo (`Base Bid Price`). L'obiettivo è andare oltre le medie e capire la *forma* e la *struttura* delle distribuzioni finanziarie, evitando le trappole statistiche come quelle del **Quartetto di Anscombe**.

In [ ]:
class FinancialAnalyzer(BasePlotter):
    """Analisi delle distribuzioni finanziarie e relazioni prezzo-criteri."""

    def plot_price_distribution_log(self, df: pd.DataFrame):
        """Genera istogramma prezzi con scala logaritmica (Matplotlib)."""
        fig, ax = plt.subplots(figsize=(12, 7))
        
        prices = df[Config.COL_PRICE][df[Config.COL_PRICE] > 0]
        mean_val = prices.mean()
        median_val = prices.median()
        
        ax.hist(prices, bins=np.logspace(np.log10(prices.min()), np.log10(prices.max()), 80),
                color='#3498db', alpha=0.7, edgecolor='black', linewidth=0.5, label='Distribuzione')
        
        ax.axvline(mean_val, color='#E74C3C', linestyle='--', linewidth=2, label=f'Media: €{mean_val:,.0f}')
        ax.axvline(median_val, color='#2ECC71', linestyle='--', linewidth=2, label=f'Mediana: €{median_val:,.0f}')
        
        ax.set_xscale('log')
        ax.set_title("Distribuzione Prezzo Base (Scala Logaritmica)", fontsize=14, weight='bold')
        ax.set_xlabel("Prezzo Base (€, scala log)", fontsize=12)
        ax.set_ylabel("Conteggio Contratti", fontsize=12)
        ax.grid(True, alpha=0.3, linestyle='--')
        ax.legend()
        
        plt.tight_layout()
        plt.show()
        self._save(fig, '17_price_distribution_log.png')

    def plot_price_by_criteria_box(self, df: pd.DataFrame):
        """Genera box plot confronto prezzi per criterio (Seaborn)."""
        if Config.COL_AWARD not in df.columns: 
            print("Colonna criterio aggiudicazione non trovata")
            return
        
        fig, ax = plt.subplots(figsize=(12, 7))
        
        # Box plot con palette professionale (blu/rosso/grigio)
        df_plot = df[df[Config.COL_PRICE] > 0].copy()
        colors = ['#3498DB', '#E74C3C', '#95A5A6'][:df_plot[Config.COL_AWARD].nunique()]
        
        sns.boxplot(data=df_plot, x=Config.COL_AWARD, y=Config.COL_PRICE, 
                    palette=colors, ax=ax, notch=True, linewidth=1.5)
        
        ax.set_yscale('log')
        ax.set_title("Confronto Prezzi per Criterio di Aggiudicazione (Scala Log)", 
                    fontsize=15, weight='bold', pad=15)
        ax.set_xlabel("Criterio Aggiudicazione", fontsize=12, weight='bold')
        ax.set_ylabel("Prezzo Base (€, scala log)", fontsize=12, weight='bold')
        ax.tick_params(axis='x', rotation=15)
        ax.grid(True, alpha=0.3, linestyle='--', axis='y')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        
        # Legenda con nomi criteri
        legend_labels = [f'Criterio {i}' for i in sorted(df_plot[Config.COL_AWARD].unique())]
        ax.legend(handles=[plt.Rectangle((0,0),1,1, color=c) for c in colors[:len(legend_labels)]], 
                 labels=legend_labels, loc='upper right', frameon=True, fontsize=10)
        
        plt.tight_layout()
        plt.show()
        self._save(fig, '18_price_by_criteria_box.png')

    def plot_price_intensity(self, df: pd.DataFrame):
        """Genera joint plot KDE per analisi densità bivariata prezzo-intensità con legenda."""
        if Config.COL_PRICE_DAY not in df.columns: 
            print("Colonna Price per Day non trovata")
            return
            
        plot_data = df[(df[Config.COL_PRICE] > 1) & (df[Config.COL_PRICE_DAY] > 1)]
        
        g = sns.jointplot(
            data=plot_data, x=Config.COL_PRICE, y=Config.COL_PRICE_DAY,
            kind="kde", fill=True, cmap='Blues', height=10,
            log_scale=(True, True)
        )
        g.fig.suptitle(
            "Intensità Economica: Prezzo Totale vs Prezzo/Giorno", 
            y=1.02, fontweight='bold', fontsize=16
        )
        g.set_axis_labels(
            "Prezzo Totale (€, log)", 
            "Costo Giornaliero (€/giorno, log)",
            fontsize=14
        )
        
        # Aggiungi testo esplicativo come legenda
        g.ax_joint.text(
            0.05, 0.95, 
            'Zone scure = Alta densità contratti\nZone chiare = Bassa densità contratti',
            transform=g.ax_joint.transAxes,
            fontsize=11,
            verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor='#2C3E50', linewidth=1.5),
            weight='bold'
        )
        
        plt.show()
        self._save(g.fig, '19_price_intensity_kde.png')

    def plot_budget_treemap(self, df: pd.DataFrame):
        if Config.COL_AWARD not in df.columns: 
            print("Colonna criterio aggiudicazione non trovata")
            return

        df_tree = df.groupby([Config.COL_DISTRICT, Config.COL_AWARD])[Config.COL_PRICE].sum().reset_index()
        df_tree = df_tree[df_tree[Config.COL_PRICE] > 0]

        district_totals = df_tree.groupby(Config.COL_DISTRICT)[Config.COL_PRICE].sum()
        df_tree['Pct_in_District'] = df_tree.apply(
            lambda row: row[Config.COL_PRICE] / district_totals[row[Config.COL_DISTRICT]] * 100, axis=1
        )

        df_tree['Label'] = df_tree.apply(
            lambda row: f"{row[Config.COL_DISTRICT]}<br>{row[Config.COL_AWARD]} ({row['Pct_in_District']:.1f}%)",
            axis=1
        )

        fig = px.treemap(
            df_tree,
            path=[px.Constant("Portogallo"), Config.COL_DISTRICT, Config.COL_AWARD],
            values='Pct_in_District',
            color='Pct_in_District',
            color_continuous_scale='RdBu',
            title="Allocazione Percentuale Budget: Distretto > Criterio Aggiudicazione",
            hover_data={
                'Pct_in_District': ':.1f'
            }
        )
        fig.update_traces(
            texttemplate="<b>%{label}</b><br>%{value:.1f}%",
            textposition="middle center"
        )
        fig.update_layout(
            height=800,
            font=dict(family="Arial", size=11)
        )
        fig.write_html(Config.PLOTS_DIR / '20_budget_treemap.html')
        fig.show()

    def plot_price_by_duration_violin(self, df: pd.DataFrame):
        """
        Genera un violin plot per analizzare la distribuzione dei prezzi
        in base alle categorie di durata del progetto.
        """
        if Config.COL_DEADLINE not in df.columns or Config.COL_PRICE not in df.columns:
            print("Colonne necessarie (Deadline, Price) per analisi prezzo/durata non trovate")
            return

        df_plot = df[[Config.COL_DEADLINE, Config.COL_PRICE]].dropna()
        df_plot = df_plot[(df_plot[Config.COL_PRICE] > 1) & (df_plot[Config.COL_DEADLINE] > 1)]

        # Crea categorie durata
        duration_bins = [0, 90, 180, 365, 730, float('inf')]
        duration_labels = ['<3 mesi', '3-6 mesi', '6-12 mesi', '1-2 anni', '>2 anni']
        
        df_plot['Duration_Category'] = pd.cut(
            df_plot[Config.COL_DEADLINE],
            bins=duration_bins,
            labels=duration_labels,
            right=True # Assicura che 0 sia incluso nel primo bin se right=False non è specificato
        )
        
        # Rimuovi eventuali categorie che non hanno dati dopo il cut
        df_plot.dropna(subset=['Duration_Category'], inplace=True)
        
        # Ordina le categorie per il plot
        active_labels = [label for label in duration_labels if label in df_plot['Duration_Category'].unique()]
        
        if not active_labels:
            print("Nessun dato trovato nelle categorie di durata definite.")
            return

        # Verifica distribuzione nelle categorie
        print("\nDistribuzione contratti per categoria durata:")
        print(df_plot['Duration_Category'].value_counts().loc[active_labels])

        fig, ax = plt.subplots(figsize=(14, 8))
        
        # Palette blu sfumato
        colors = ['#D1E5F0', '#92C5DE', '#4393C3', '#2166AC', '#053061']
        
        sns.violinplot(
            data=df_plot, 
            x='Duration_Category', 
            y=Config.COL_PRICE,
            order=active_labels, # Assicura l'ordine corretto sull'asse X
            palette=colors[:len(active_labels)], 
            inner='box', 
            ax=ax, 
            linewidth=1.5, 
            cut=0, 
            scale='width'
        )
        
        ax.set_yscale('log')
        ax.set_title("Distribuzione Prezzo per Durata Progetto", fontsize=16, weight='bold', pad=20)
        ax.set_xlabel("Categoria Durata", fontsize=13, weight='bold')
        ax.set_ylabel("Prezzo Base (€, scala log)", fontsize=13, weight='bold')
        ax.tick_params(axis='x', rotation=45, labelsize=11)
        ax.tick_params(axis='y', labelsize=11)
        ax.grid(True, alpha=0.3, linestyle='--', axis='y')
        sns.despine(ax=ax) # Rimuove spine top/right

        # Aggiungi conteggi su ogni violin
        # Calcola la posizione Y massima per il testo (in coordinate logaritmiche)
        ymin, ymax = ax.get_ylim()
        text_y_pos = ymax * 0.9 
        
        for i, cat in enumerate(active_labels):
            count = len(df_plot[df_plot['Duration_Category'] == cat])
            if count > 0:
                ax.text(i, text_y_pos, f'n={count}', 
                       ha='center', fontsize=10, weight='bold', color='#2C3030',
                       bbox=dict(boxstyle='round', facecolor='white', alpha=0.7, edgecolor='gray'))
        
        plt.tight_layout()
        plt.show()
        
        self._save(fig, '17b_price_by_duration_violin.png')
    
    def plot_award_criteria_distribution(self, df: pd.DataFrame):
        import matplotlib.patheffects as path_effects
        
        if Config.COL_AWARD not in df.columns:
            print("Colonna criteri aggiudicazione non trovata")
            return

        award_dist = df[Config.COL_AWARD].value_counts()
        
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7), gridspec_kw={'width_ratios': [1.2, 1]})
        
        # --- USA LA LISTA DI COLORI ORIGINALE ---
        # Si basa sull'ordine di value_counts(), che è quello che vuoi
        colors = ['#2C3E50', '#E74C3C', '#3498DB'][:len(award_dist)]
        
        ax1.barh(award_dist.index, award_dist.values, color=colors, edgecolor='white', linewidth=2)
        ax1.set_xlabel("Numero Contratti", fontsize=12, weight='bold')
        ax1.set_ylabel("Criterio Aggiudicazione", fontsize=12, weight='bold')
        ax1.set_title("Quantità Assolute per Criterio", fontsize=14, weight='bold', pad=15)
        ax1.grid(True, alpha=0.3, linestyle='--', axis='x')
        sns.despine(ax=ax1, top=True, right=True)
        
        # --- LOOP DI ETICHETTATURA CORRETTO ---
        for bar in ax1.patches:
            val = bar.get_width()
            y_pos = bar.get_y() + bar.get_height() / 2
            
            ax1.text(val + max(award_dist.values) * 0.02,
                     y_pos,
                     f'{int(val):,}',
                     va='center', 
                     fontsize=11, 
                     weight='bold', 
                     color='#2C3E50')

        wedges, texts, autotexts = ax2.pie(
            award_dist, 
            labels=award_dist.index,
            autopct='%1.1f%%',
            startangle=90,
            colors=colors, # I colori ora sono corretti anche qui
            wedgeprops=dict(edgecolor='white', linewidth=2.5),
            textprops={'fontsize': 11, 'weight':'bold'}
        )
        
        for autotext in autotexts:
            autotext.set_color('white')
            autotext.set_fontsize(12)
            autotext.set_weight('bold')
            autotext.set_path_effects([
                path_effects.withStroke(linewidth=2, foreground='black')
            ])
        
        ax2.set_title("Distribuzione Percentuale", fontsize=14, weight='bold', pad=15)
        plt.tight_layout()
        plt.show()
        self._save(fig, '20b_award_criteria_pie.png')

# --- INIZIALIZZAZIONE ANALYZER ---
fin_analyzer = FinancialAnalyzer()

### 4.1 Grafico 1: Distribuzione Prezzo Base (Scala Logaritmica)

**Scelta Tecnica: Istogramma Log-Scale**

Distribuzione estremamente right-skewed (range: €10k - €100M+). Scala lineare comprimerebbe 99% dei dati in poche barre, rendendo l'analisi impossibile.

**Design Space:**
* **Asse X:** Prezzo Base € (Quantitativo) → scala logaritmica
* **Asse Y:** Conteggio Contratti (Quantitativo)
* **Segni:** Barre (bin logaritmici equidistanti)

Trasformazione log "comprime" ordini di grandezza, visualizzando simultaneamente micro-contratti (€10k) e mega-progetti (€100M).

---

**Interpretazione**

**Distribuzione Log-Normale con Bimodalità**

*Moda Primaria (€200k-€400k):*
- Picco centrale ~375 contratti per bin
- Appalti ordinari: manutenzioni, lavori civili standard
- Bulk del mercato operativo

*Coda Lunga Destra (>€5M):*
- Decadimento lento fino a €100M
- Mega-progetti infrastrutturali (Beja-style)
- Numericamente rari ma dominanti in valore

*Coda Sinistra (<€50k):*
- Micro-appalti operativi
- Forniture, servizi minori

**Asimmetria Estrema Confermata**
- Range: 4+ ordini di grandezza (10^4 → 10^8)
- Distribuzione non normale: log-trasformazione la "normalizza"
- Media >> Mediana (forte skew destro)

**Insight Quartetto di Anscombe**
Bimodalità e asimmetria invisibili con sole statistiche di riepilogo (media €0.8M). Visualizzazione rivela struttura reale: due mercati sovrapposti (operativo + strategico).

**Conclusione:** Mercato con distribuzione power-law. Pochi contratti giganti spostano media, ma massa è concentrata su piccoli/medi appalti. Conferma dualità strutturale identificata nelle analisi geografiche.

In [ ]:
fin_analyzer.plot_price_distribution_log(df_master)

### 4.2 Grafico 2: Distribuzione Prezzo per Durata Progetto (Violin Plot)

**Scelta Tecnica: Violin Plot con Scala Log**

Combina box plot (statistiche robuste) + KDE (forma distribuzione) per confronto multi-categoria.

**Design Space:**
* **Asse X:** Categoria Durata (Ordinale: <3m, 3-6m, 6-12m, 1-2y, >2y)
* **Asse Y:** Prezzo Base € (Quantitativo, scala log)
* **Violino:** Larghezza = densità distribuzione (KDE specchiata)
* **Box interno:** Mediana (linea), quartili (box), outlier (punti)

---

**Interpretazione**

**Correlazione Durata-Prezzo Progressiva**

*Progetti Brevi (<3 mesi, n=1272):*
- Mediana: ~€200k
- Distribuzione stretta, concentrata €100k-€500k
- Forma simmetrica in log-scale
- Appalti operativi veloci

*Progetti Medi (3-6 mesi, n=1863):*
- Mediana: ~€300k
- Distribuzione si allarga verso l'alto
- Prime code pesanti (outlier fino €30M)

*Progetti Lunghi (6-12 mesi, n=1353):*
- Mediana: ~€500k
- Distribuzione più piatta e larga
- Multimodalità emergente
- Range: €50k - €20M

*Progetti Complessi (1-2 anni, n=344):*
- Mediana: ~€2M (4x rispetto a <3m)
- Distribuzione molto larga e bimodale
- Outlier estremi fino €100M
- Dominano mega-progetti infrastrutturali

*Progetti Pluriennali (>2 anni, n=0):*
- Dataset vuoto: assenza totale progetti oltre 2 anni
- Possibile limite normativo o pratico del mercato

**Insight Forma Distribuzione**
- Durata breve: distribuzione normale (in log)
- Durata lunga: distribuzione power-law con bimodalità
- Variabilità aumenta esponenzialmente con durata

**Conclusione:** Durata è proxy forte per complessità e valore. Progetti lunghi = due mercati sovrapposti (manutenzioni pluriennali + infrastrutture strategiche). Assenza >2 anni suggerisce segmentazione normativa o frammentazione temporale di mega-progetti.

In [ ]:
fin_analyzer.plot_price_by_duration_violin(df_master)

### 4.3 Grafico 3: Confronto Prezzi per Criterio di Aggiudicazione (Box Plot)

**Scelta Tecnica: Box Plot con Scala Log**

Confronta distribuzioni prezzi tra i 3 criteri di aggiudicazione identificati nell'analisi temporale (Criterio 0: dominante pre-2014, Criterio 1 e 2: post-2014).

**Design Space:**
* **Asse X:** Criterio Aggiudicazione (Categorico: 0, 1, 2)
* **Asse Y:** Prezzo Base € (Quantitativo, scala log)
* **Box:** Quartili (Q1-Q3), mediana (linea centrale)
* **Whiskers:** Range interquartile esteso (1.5×IQR)
* **Outlier:** Punti individuali oltre whiskers

---

**Interpretazione**

**Differenze Mediane Significative**

*Criterio 0 (pre-2014, lowest price):*
- Mediana: ~€500k
- IQR: €300k - €1.2M
- Outlier estremi fino €100M
- Distribuzione più ampia tra i tre

*Criterio 1 (post-2014):*
- Mediana: ~€400k (leggermente inferiore a C0)
- IQR: €250k - €800k
- Outlier fino €30M
- Concentrazione maggiore su valori medio-bassi

*Criterio 2 (post-2014):*
- Mediana: ~€250k (la più bassa)
- IQR: €150k - €500k
- Outlier fino €15M
- Distribuzione più compressa, meno mega-progetti

**Cambiamento Post 2014**
- Criterio 0 aveva range più ampio e outlier più estremi
- Criteri 1 e 2 mostrano maggiore standardizzazione
- Riduzione valore mediano generale post-2014
- Frammentazione: più contratti piccoli/medi

**Outlier Pattern**
- Tutti i criteri hanno code lunghe estreme
- C0 ha concentrazione maggiore di mega-progetti (>€10M)
- C2 più "pulito": meno outlier estremi

**Conclusione:** Il cambio dal 2014 ha redistribuito non solo il volume ma anche la tipologia contrattuale. Criterio 0 (abolito) era associato a progetti più grandi e variegati. I nuovi criteri mostrano segmentazione: C1 copre range intermedio, C2 concentrato su piccoli/medi appalti.

In [ ]:
fin_analyzer.plot_price_by_criteria_box(df_master)

### 4.4 Grafico 4: Intensità Economica - Prezzo Totale vs Costo Giornaliero

**Scelta Tecnica: 2D Density Plot (Hexbin/Contour) con Marginal Distributions**

Visualizzazione bivariata avanzata per esplorare relazione tra due variabili quantitative continue su scala log.

**Design Space:**
* **Plot Centrale:** 2D KDE (contour density)
  - Asse X: Prezzo Totale € (log)
  - Asse Y: Costo €/Giorno (log)
  - Colore: Densità contratti (blu scuro = alta concentrazione)
* **Marginal Plots:** KDE 1D per distribuzioni univariate
  - Top: Distribuzione Prezzo Totale
  - Destra: Distribuzione Costo/Giorno

Tecnica superiore a scatter plot per >4000 punti: evita overplotting, rivela concentrazioni.

---

**Interpretazione**

**Correlazione Positiva Debole con Cluster Principale**

*Cluster Dominante (zona blu scuro):*
- Prezzo: €100k - €500k
- Costo/giorno: €1k - €3k
- ~60% dei contratti
- Progetti standard: durata 100-300 giorni, intensità media

**Due Traiettorie Distinte**

*Traiettoria Efficienza (diagonale ripida):*
- Prezzo aumenta più velocemente di costo/giorno
- Progetti lunghi (>1 anno) con economia di scala
- Costo/giorno relativamente basso rispetto a totale

*Traiettoria Intensità (diagonale piatta):*
- Costo/giorno alto, prezzo totale moderato
- Progetti brevi e intensivi
- Alta concentrazione risorse temporale

**Outlier Pattern**

*Mega-Progetti (estremo destro):*
- Prezzo >€100M, costo/giorno €50k-€100k
- Dispersione verticale alta: intensità molto variabile
- Pochi casi (Beja-style), bassa densità

*Alta Intensità (estremo alto):*
- Costo/giorno >€10k con prezzo totale contenuto
- Progetti brevi ad alta intensità operativa
- Emergenze, interventi urgenti

**Distribuzioni Marginali**

*Prezzo Totale (top):*
- Log-normale centrata su €200k
- Coda destra lunga fino €100M+

*Costo/Giorno (destra):*
- Bimodale: picco primario ~€1.5k, secondario ~€5k
- Riflette due segmenti: operativo ordinario vs intensivo

**Conclusione:** Relazione non lineare complessa. Maggioranza contratti in "sweet spot" €100k-€500k con €1-3k/giorno. Dispersione aumenta con prezzo totale: mega-progetti mostrano intensità altamente variabile (da economia di scala a emergenze). Assenza zona "alto prezzo + basso costo/giorno" suggerisce limite minimo di intensità per grandi opere.

In [ ]:
# Analisi densità bivariata: relazione tra prezzo totale e intensità economica giornaliera
fin_analyzer.plot_price_intensity(df_master)

### 4.5 Grafico 5: Treemap Gerarchico - Allocazione Budget per Distretto e Criterio

**Scelta Tecnica: Treemap Interattivo (Space-Filling Hierarchy)**

Visualizzazione gerarchica a due livelli (Distretto → Criterio) con interattività nativa per esplorazione drill-down.

**Design Space:**
* **Livello 1:** Rettangoli distretto, area = budget totale
* **Livello 2:** Suddivisioni criterio, area = budget per criterio  
* **Colore:** Saturazione = % budget distretto su criterio
* **Interattività:** Click = zoom gerarchico, hover = details-on-demand

**Implementazione Mantra Shneiderman:**
- **Overview:** Vista completa distribuzione nazionale
- **Zoom:** Click su distretto → vista interna composizione criteri
- **Details-on-demand:** Hover → valori esatti e percentuali

---

**Interpretazione**

**Dominanza Geografica con Specializzazioni**

*Beja - Monolite Pre-Riforma:*
- Rettangolo dominante (93% C0)
- Blu scuro saturo: nessuna diversificazione
- Conferma isolamento temporale (contratti legacy)

*Porto/Lisboa - Transizione Bilanciata:*
- Mix equilibrato C0/C1/C2 (35-41% ciascuno)
- Adozione progressiva nuovi criteri
- Mercati dinamici post-2014

*Periferie - Persistenza Legacy:*
- Madeira (78% C0), Évora (75% C0), Vila Real (74% C0)
- Colori saturi: bassa frammentazione
- Contratti pluriennali ancora attivi

**Pattern Drill-Down (Funzionalità Interattiva)**

Zoom su singolo distretto rivela:
- Proporzioni interne criteri
- Identificazione criterio dominante
- Confronto con distribuzione nazionale

**Vantaggio vs Visualizzazioni Statiche**
- Treemap comprime 3 dimensioni (distretto, criterio, budget) in 2D
- Interattività elimina necessità di grafici separati
- Supporta esplorazione esplorativa senza perdere contesto

**Conclusione:** Treemap interattivo ottimale per gerarchie budget. Rivela simultaneamente concentrazione geografica (Beja domina spazio) e specializzazione criteri (saturazione colore). Funzionalità zoom implementa "progressive disclosure" per analisi multi-scala.

In [ ]:
# Visualizzazione gerarchica allocazione budget: Distretto > Criterio di Aggiudicazione
fin_analyzer.plot_budget_treemap(df_master)

### 4.6 Grafico 6: Distribuzione Criteri di Aggiudicazione (Bar + Pie)

**Scelta Tecnica: Visualizzazione Duale per Analisi e Comunicazione**

Combina bar chart (precisione) e pie chart (comunicazione part-to-whole).

**Design Space:**
* **Bar Chart:** Lunghezza → N. Contratti assoluti
* **Pie Chart:** Angolo/Area → % distribuzione

---

**Interpretazione**

**Criterio 2 (grigio scuro) Dominante:**
- 2,221 contratti (46%)
- Quasi metà del totale dataset
- Criterio più utilizzato

**Criterio 1 (blu) Intermedio:**
- 1,419 contratti (24.7%)
- Circa un quarto

**Criterio 0 (rosso) Minore:**
- ~1,400 contratti (29.4%)
- Quota intermedia

**Distribuzione Non Uniforme**
- Forte sbilanciamento verso Criterio 2
- Nessuno dei tre criteri sotto 20%
- Tutti e tre rappresentati significativamente nel dataset

**Vantaggio Duale**
- Bar: confronto quantitativo preciso delle magnitudini
- Pie: percezione immediata "C2 = quasi metà"

**Conclusione:** Criterio 2 emerge come standard dominante (46%), con Criterio 0 e 1 che si dividono la quota restante in modo relativamente equilibrato.

In [ ]:
fin_analyzer.plot_award_criteria_distribution(df_master)

## 5. Analisi Testuale e Semantica (Zoom sul "Cosa")

Dopo aver analizzato *Quanto*, *Quando* e *Dove*, ora "zoomiamo" sul *Cosa*: qual è l'oggetto dei contratti? In questa sezione, visualizziamo i risultati della nostra pipeline NLP (Fase 1, Sezione 7).

L'obiettivo è visualizzare i **Dati 1D-Lineari** (testo) che abbiamo trasformato in *feature* strutturate.

In [ ]:
from wordcloud import WordCloud

class TextAnalyzer(BasePlotter):
    """Analisi NLP per estrazione temi, clustering semantico e visualizzazione keyword."""
    
    def plot_keyword_stats(self, df: pd.DataFrame):
        """Genera bar chart + pie chart per top keyword (Matplotlib/Seaborn)."""
        kw_cols = [c for c in df.columns if c.startswith('cpvs_keyword_')]
        if not kw_cols: 
            print("Colonne keyword non trovate")
            return

        kw_counts = df[kw_cols].sum().sort_values(ascending=True)
        kw_counts.index = kw_counts.index.str.replace('cpvs_keyword_', '').str.replace('_', ' ')

        # Bar Chart orizzontale con gradient professionale
        fig1, ax1 = plt.subplots(figsize=(12, max(8, len(kw_counts)*0.3)))
        # Gradient da blu scuro a blu chiaro (no verdini)
        colors = plt.cm.Blues(np.linspace(0.5, 0.95, len(kw_counts)))
        bars = ax1.barh(kw_counts.index, kw_counts.values, color=colors, edgecolor='#2C3E50', linewidth=0.8)
        
        # Valori alla fine delle barre
        for i, (idx, val) in enumerate(kw_counts.items()):
            ax1.text(val + max(kw_counts.values)*0.01, i, f'{int(val)}', 
                    va='center', fontsize=10, weight='bold', color='#2C3E50')
        
        ax1.set_title("Frequenza Top Keyword nei Contratti (TF-IDF)", fontsize=15, weight='bold', pad=15)
        ax1.set_xlabel("Occorrenze Aggregate", fontsize=12, weight='bold')
        ax1.set_ylabel("Termini Estratti", fontsize=12, weight='bold')
        ax1.grid(True, alpha=0.3, linestyle='--', axis='x')
        ax1.spines['top'].set_visible(False)
        ax1.spines['right'].set_visible(False)
        plt.tight_layout()
        plt.show()
        self._save(fig1, '21a_keyword_frequency_bar.png')

        # Donut Chart con palette moderna
        fig2, ax2 = plt.subplots(figsize=(11, 8))
        # Palette categorica moderna (no verdini)
        colors_pie = ['#2C3E50', '#E74C3C', '#3498DB', '#F39C12', '#9B59B6', 
                     '#1ABC9C', '#34495E', '#E67E22', '#95A5A6', '#16A085'][:len(kw_counts)]
        
        wedges, texts, autotexts = ax2.pie(
            kw_counts.values, 
            labels=kw_counts.index,
            autopct='%1.1f%%',
            startangle=90,
            colors=colors_pie,
            wedgeprops=dict(width=0.4, edgecolor='white', linewidth=2.5),
            textprops={'fontsize': 10, 'weight': 'bold'}
        )
        
        # Percentuali con contorno per leggibilità
        import matplotlib.patheffects as path_effects
        for autotext in autotexts:
            autotext.set_color('white')
            autotext.set_fontsize(11)
            autotext.set_weight('bold')
            autotext.set_path_effects([
                path_effects.withStroke(linewidth=2, foreground='black')
            ])
        
        # Legenda a destra
        ax2.legend(kw_counts.index, loc='center left', bbox_to_anchor=(1, 0.5), 
                  fontsize=10, frameon=True, title='Keywords', title_fontsize=11)
        
        ax2.set_title("Distribuzione Percentuale Keyword", fontsize=15, weight='bold', pad=20)
        plt.tight_layout()
        plt.show()
        self._save(fig2, '21b_keyword_distribution_pie.png')

    def plot_keyword_by_criteria(self, df: pd.DataFrame, top_n: int = 10):
        if Config.COL_AWARD not in df.columns:
            print("Colonna criteri aggiudicazione non trovata")
            return
            
        kw_cols = [c for c in df.columns if c.startswith('cpvs_keyword_')]
        if not kw_cols:
            print("Colonne keyword non trovate")
            return

        top_kw = df[kw_cols].sum().nlargest(top_n).index.tolist()

        try:
            pivot = df.groupby(Config.COL_AWARD)[top_kw].sum().T
            pivot.index = pivot.index.str.replace('cpvs_keyword_', '').str.replace('_', ' ')
            pivot.index.name = 'Keyword'
            pivot.columns.name = 'Criterio'
            pivot = pivot[pivot.sum(axis=1) > 0]
            
            if pivot.empty:
                print("Nessun dato per cross-analysis keyword-criteri")
                return
        except Exception as e:
            print(f"Errore durante l'aggregazione dei dati: {e}")
            return

        fig, ax = plt.subplots(figsize=(14, 8))
        
        legend_labels = pivot.columns.tolist()
        colors_criteria = ['#3498DB', '#E74C3C', '#95A5A6', '#F39C12', '#9B59B6'][:len(legend_labels)]
        
        pivot.plot(kind='bar', ax=ax, width=0.8, color=colors_criteria, edgecolor='#2C3E50', linewidth=0.8)
        
        ax.set_title(f"Top {top_n} Keyword per Criterio di Aggiudicazione", fontsize=15, weight='bold', pad=15)
        ax.set_xlabel("Keyword", fontsize=12, weight='bold')
        ax.set_ylabel("Occorrenze", fontsize=12, weight='bold')
        
        ax.legend(legend_labels, title='Criterio Aggiudicazione', 
                  loc='upper left', bbox_to_anchor=(1, 1), fontsize=10, 
                  frameon=True, borderpad=1)
        
        plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
        ax.grid(True, alpha=0.3, linestyle='--', axis='y')
        sns.despine(ax=ax, top=True, right=True)
        
        plt.tight_layout()
        plt.show()
        self._save(fig, '21c_keyword_by_criteria.png')

    def plot_wordcloud(self, df: pd.DataFrame, text_col: str):
        """Genera word cloud da testi preprocessati (lemmatized, stop words removed)."""
        if text_col not in df.columns: 
            print(f"Colonna {text_col} non trovata")
            return
            
        text = " ".join(df[text_col].dropna().astype(str))
        
        if len(text.strip()) < 100:
            print("Testo insufficiente per word cloud")
            return
            
        wc = WordCloud(
            width=1800, height=900, 
            background_color='black',  # Fondo nero per contrasto
            colormap='rainbow',  # Colori multipli ben leggibili
            max_words=200,
            relative_scaling=0.5,
            min_font_size=10,
            prefer_horizontal=0.7,
            contour_width=2,
            contour_color='white'  # Contorno bianco su fondo nero
        ).generate(text)
        
        fig, ax = plt.subplots(figsize=(18, 9))
        fig.patch.set_facecolor('black')  # Fondo figura nero
        ax.set_facecolor('black')  # Fondo axes nero
        ax.imshow(wc, interpolation='bilinear')
        ax.axis("off")
        ax.set_title(f"Word Cloud: {text_col}", fontsize=20, fontweight='bold', 
                    pad=20, color='white')  # Titolo bianco su fondo nero
        plt.tight_layout()
        plt.show()
        self._save(fig, '22_wordcloud.png')

    def plot_semantic_clusters(self, df: pd.DataFrame):
        """Scatter plot 2D dei cluster semantici da embeddings + PCA + K-Means."""
        req = ['semantic_x', 'semantic_y', 'semantic_cluster']
        if not all(c in df.columns for c in req): 
            print("Colonne semantic_x/y/cluster non trovate")
            return
            
        df_plot = df.dropna(subset=req).copy()
        df_plot['semantic_cluster'] = df_plot['semantic_cluster'].astype(str)
        
        fig = px.scatter(
            df_plot, x='semantic_x', y='semantic_y', 
            color='semantic_cluster',
            hover_data=[Config.COL_CPVS, Config.COL_PRICE] if Config.COL_CPVS in df.columns else None,
            title="Mappa Semantica Contratti: Cluster Tematici (PCA + K-Means)",
            labels={
                'semantic_x': 'Componente Principale 1',
                'semantic_y': 'Componente Principale 2',
                'semantic_cluster': 'Cluster'
            },
            color_discrete_sequence=px.colors.qualitative.Bold, 
            opacity=0.7
        )
        fig.update_layout(
            plot_bgcolor='rgba(248,249,250,1)',
            height=700,
            font=dict(family="Arial", size=12),
            legend_title_text='Cluster Semantico'
        )
        fig.update_traces(marker=dict(size=8, line=dict(width=0.5, color='white')))
        fig.write_html(Config.PLOTS_DIR / '23_semantic_clusters.html')
        fig.show()

# --- INIZIALIZZAZIONE ANALYZER ---
text_analyzer = TextAnalyzer()

### 5.1 Grafico 1: Frequenza Top Keyword nei Contratti (TF-IDF)

**Scelta Tecnica: Bar Chart Orizzontale per Ranking Testuale**

Visualizza risultati analisi NLP (TF-IDF) su descrizioni contratti, implementando principio **"Analyze First"** (Keim): preprocessing automatico identifica pattern semantici prima della visualizzazione.

**Design Space:**
* **Asse Y:** Termini estratti (Nominale, ordinati per rilevanza)
* **Asse X:** Occorrenze aggregate (Quantitativo)
* **Canale:** Lunghezza = importanza termine (canale più efficace per ranking, gerarchia Mackinlay)

**Collegamento Teorico - Visual Readability (Oelke et al., 2010):**

Approccio analogo a **VisRA**: invece di estrarre feature linguistiche di leggibilità (complessità vocabolario, lunghezza frasi) da testi letterari, estraiamo feature semantiche (TF-IDF) da corpus contratti per identificare temi distintivi. Entrambi:
1. **Feature Selection semi-automatica** da corpus testuale 1D
2. **Visualizzazione aggregata** per supportare analisi esplorativa
3. **Obiettivo:** rivelare pattern non evidenti in lettura sequenziale

TF-IDF filtra stopwords generiche, amplifica termini distintivi per documento/categoria.

---

**Interpretazione**

**Dominanza Termini Generici Edili**

*"work" (4620):*
- Baseline linguistico settore
- Termine omnicomprensivo operativo

*"construction" + "construction work" (~3676 combinati):*
- Core semantico dataset
- Lavori edili/civili dominanti

**Cluster Semantici Specializzati**

*Infrastrutture stradali (~2195 occorrenze):*
- "surface" (751), "road" (748), "surface work" (696)
- Manutenzione stradale come categoria primaria

*Riqualificazione edificato esistente (~851):*
- "refurbishment" (470), "refurbishment work" (381)
- Prevalenza manutenzione su nuove costruzioni

*Altre specializzazioni:*
- "building" (405), "pipeline" (399)

**Pattern Linguistico Standardizzato**

- Prevalenza bigrammi "X work"
- Terminologia tecnica normalizzata
- Lingua descrittiva vs commerciale

**Assenza Mega-Progetti in Frequenza**

- Nessun termine: dighe, ponti, autostrade, porti
- Mega-progetti Beja numericamente irrilevanti per TF-IDF
- Conferma bulk operativo del mercato

**Conclusione:** TF-IDF rivela vocabolario dominato da manutenzione ordinaria. Visualizzazione con bar chart sfrutta lunghezza (canale più efficace) per ranking preciso. Analogia con VisRA: entrambi usano feature extraction + InfoVis per trasformare dati testuali 1D in insight azionabili.

In [ ]:
# Analisi quantitativa frequenza keyword tramite TF-IDF
text_analyzer.plot_keyword_stats(df_master)

### 5.2 Grafico 2: Top 10 Keyword per Criterio di Aggiudicazione

**Scelta Tecnica: Grouped Bar Chart (Confronto Multi-Categorico)**

Visualizzazione cross-dimensionale: Keyword × Criterio × Frequenza.

**Design Space:**
* **Asse X:** Keyword (Nominale, 10 top termini)
* **Asse Y:** Occorrenze (Quantitativo)
* **Colore (Hue):** Criterio Aggiudicazione (Nominale: 0=blu, 1=rosso, 2=grigio)
* **Raggruppamento:** Barre affiancate per confronto diretto stesso termine

Tecnica superiore a stacked bars per confronti precisi tra categorie (evita somme cumulative che distorcono percezione).

---

**Interpretazione**

**Omogeneità Linguistica Tra Criteri**

Tutti i top 10 termini appaiono in tutti e 3 i criteri con frequenze comparabili. Nessuna specializzazione lessicale evidente per criterio.

**Dominanza Criterio 2 (grigio) Confermata**

Per quasi tutti i termini, barra grigia è la più alta:
- "work": C2 domina (~2100) vs C0 (~1350) e C1 (~1150)
- "construction": C2 (~750) > C0 (~700) > C1 (~470)
- Pattern coerente con distribuzione generale 46% C2

**Eccezioni Criterio 0 (blu)**

Criterio 0 supera gli altri in:
- "construction work": C0 (~700) > C2 (~650)
- "construction": C0 (~700) leggermente > C2 (~750) [verificare visivamente]

Possibile preferenza legacy per terminologia "construction" nel periodo pre-2014.

**Criterio 1 (rosso) Sempre Terzo**

Per tutti i 10 termini, C1 ha frequenza minore. Coerente con quota 24.7% del totale.

**Assenza Specializzazione Semantica**

- Nessun termine "esclusivo" di un criterio
- Vocabolario standardizzato indipendente da metodo aggiudicazione
- Terminologia descrittiva tecnica, non strategica

**Scala Relativa Conservata**

Ranking interno keyword consistente tra criteri:
1. "work" domina sempre
2. "construction" + "construction work" seguono
3. Termini specifici (surface, road, refurbishment) simili tra criteri

**Conclusione:** Assenza differenziazione linguistica tra criteri. Frequenze rispecchiano solo proporzioni volumetriche (46%-29%-25%). Nessun "linguaggio strategico" distinto per criterio qualità vs prezzo. Conferma natura operativa e standardizzata delle descrizioni contrattuali.

In [ ]:
# Analisi keyword per criterio aggiudicazione
text_analyzer.plot_keyword_by_criteria(df_master, top_n=10)

### 5.3 Grafico 3: Word Cloud - Overview Visuale Corpus Testuale

**Scelta Tecnica: Word Cloud (Comunicazione vs Analisi)**

Visualizzazione dati testuali 1D con limitazioni percettive consapevoli.

**Design Space:**
* **Canale:** Dimensione font = Frequenza termine (percepito come Area)
* **Colore:** Puramente estetico, nessuna codifica
* **Posizione:** Casuale/algoritmica, non significativa

**Limitazioni Percettive:**
- Area = canale impreciso (fondo gerarchia Mackinlay)
- Confronti quantitativi impossibili (quanto è più grande "construction" vs "work"?)
- Layout casuale nasconde relazioni semantiche

**Motivazione d'Uso - Beyond Memorability (Borkin et al.):**

Non inclusa per precisione analitica, ma per **impatto comunicativo** e **memorabilità**:
1. **Visual Hook:** Forma unica e distintiva facilita recall
2. **Human Recognizable Objects:** Parole leggibili = oggetti familiari
3. **Gestalt immediato:** Colpo d'occhio rivela temi dominanti senza lettura analitica
4. **Message Redundancy:** Rinforza insights già ottenuti da bar chart

Studio Borkin dimostra: visualizzazioni con forme uniche e elementi testuali hanno migliore retention anche se analiticamente inferiori.

---

**Interpretazione**

**Gerarchia Visiva Confermata**

Termini dominanti (font massimo):
- "construction", "work" (arancione/giallo)
- Conferma analisi TF-IDF quantitativa

**Cluster Semantici Percepibili**

Termini medio-grandi rivelano temi:
- **Infrastrutture:** sewage, pipeline, water, road (blu/ciano)
- **Edilizia:** building, school, refurbishment (rossi/viola)
- **Opere civili:** surface, overhaul, repair (verdi/rossi)

**Pattern Bigrammi Visibile**

Ripetizione "X work" emerge anche visivamente:
- construction work, surface work, refurbishment work, pipeline work
- Conferma standardizzazione linguistica

**Conclusione:** Word cloud efficace per comunicazione rapida e memorabilità, non per analisi precisa. Complementa bar chart: precisione quantitativa (bar) + impatto visivo (cloud). Uso giustificato da teoria memorabilità: sacrificio precisione per retention messaggio chiave ("mercato dominato da construction/work").

In [ ]:
# Generazione word cloud da testi preprocessati (lemmatized, stop words removed)
txt_col = 'Cpvs Designation_cleaned' if 'Cpvs Designation_cleaned' in df_master.columns else Config.COL_CPVS
text_analyzer.plot_wordcloud(df_master, txt_col)

### 5.4 Grafico 4: Mappa Semantica Contratti (PCA + K-Means Clustering)

**Scelta Tecnica: Scatter Plot su Spazio Latente Ridotto**

Visualizza risultati pipeline NLP completa: Embeddings → Clustering → Riduzione Dimensionale.

**Pipeline "Analyze First" (Keim):**

1. **Preprocessing:** Testo 1D → TF-IDF vectorization → spazio n-D (migliaia di dimensioni)
2. **Clustering:** K-Means su vettori TF-IDF → 5 cluster tematici
3. **Riduzione Dimensionale:** PCA (n-D → 2D) per visualizzazione
4. **Visual Mapping:** Substrato 2D semantico (non geografico)

**Design Space:**
* **Asse X:** PC1 (Componente Principale 1) - varianza massima
* **Asse Y:** PC2 (Componente Principale 2) - varianza secondaria
* **Colore (Hue):** Cluster ID (Nominale: 0,1,2,3,4)
* **Posizione:** Creata da PCA, non intrinseca ai dati (puro InfoVis)

**Motivazione Tecnica:**

Dati testuali non hanno "posizione naturale" (come coordinate geografiche). PCA costruisce substrato artificiale dove **distanza euclidea ≈ similarità semantica**. Questo è quintessenza InfoVis: creare spatial layout efficace per dati astratti (Keim, 2002).

---

**Interpretazione**

**5 Cluster Tematici Identificati**

*Cluster 4 (blu, centro-sinistra):*
- Densità massima, ~400 contratti
- Cluster "mainstream": lavori edili generici
- Baricentro vicino a origine → tema medio/generale

*Cluster 0 (verde/teal, centro-destra):*
- ~150 contratti, dispersione media
- Tema infrastrutture/civile (basato su posizione PC1 positiva)

*Cluster 2 (giallo, destra):*
- ~100 contratti, alta dispersione
- Outlier semantici sulla destra
- Possibile specializzazione tecnica (pipeline/utilities)

*Cluster 3 (rosa, alto):*
- ~15 contratti, molto dispersi
- PC2 positiva estrema = tema ortogonale al mainstream
- Possibili mega-progetti o categoria anomala

*Cluster 1 (viola, disperso):*
- ~5 contratti isolati
- True outliers semantici

**Separazione Spaziale**

- **Asse X (PC1):** Discrimina generico (sinistra) vs specializzato (destra)
- **Asse Y (PC2):** Discrimina civile standard (centro) vs anomalie (estremi)
- Cluster 4 e 0 sovrapposti parzialmente → temi correlati
- Cluster 2 ben separato → specializzazione netta

**Qualità Clustering**

- Silhouette score implicito: cluster 4 compatto (buono), cluster 3 disperso (debole)
- Presenza outlier (cluster 1) corretto: non forzati in cluster inappropriati

**Limitazione PCA**

- PC1+PC2 catturano solo frazione varianza totale (tipicamente 20-40%)
- Separazione 2D è approssimazione: cluster sovrapposti in 2D potrebbero essere separati in dimensioni superiori

**Conclusione:** Mappa rivela struttura tematica latente. Mercato dominato da cluster mainstream (blu) con nicchie specializzate (giallo/verde) e anomalie (rosa). Posizione semantica conferma: bulk operativo al centro, specializzazioni tecniche ai margini. Esempio paradigmatico InfoVis: layout artificiale (PCA) rende interpretabile spazio astratto n-D.

In [ ]:
# Visualizzazione mappa semantica: Sentence Embeddings → PCA 2D → K-Means Clustering
text_analyzer.plot_semantic_clusters(df_master)

## 6. Analisi Comparative e "Details on Demand"

Nell'ultima sezione, completiamo il mantra di Shneiderman fornendo visualizzazioni interattive e tabelle che abilitano l'esplorazione **"Details on Demand"**.

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

class ComparativeAnalyzer(BasePlotter):
    """Analisi comparative avanzate con scatter plots interattivi e tabelle metriche."""
    
    def plot_price_intensity_interactive(self, df: pd.DataFrame):
        """Scatter plot interattivo Prezzo vs Costo/Giorno con marginals."""
        if Config.COL_PRICE_DAY not in df.columns or Config.COL_PRICE not in df.columns:
            print("Colonne necessarie non trovate")
            return None
        
        df_valid = df[(df[Config.COL_PRICE] > 1) & (df[Config.COL_PRICE_DAY] > 1)].copy()
        
        if len(df_valid) > 10000:
            df_valid = df_valid.sample(n=10000, random_state=42)
        
        fig = px.scatter(
            df_valid, 
            x=Config.COL_PRICE, 
            y=Config.COL_PRICE_DAY,
            color=Config.COL_AWARD if Config.COL_AWARD in df.columns else None,
            marginal_x='histogram',
            marginal_y='histogram',
            log_x=True, 
            log_y=True,
            opacity=0.65,
            title="Intensità Economica: Prezzo Totale vs Costo Giornaliero",
            hover_data=[Config.COL_DISTRICT, Config.COL_DEADLINE] if Config.COL_DISTRICT in df.columns else None,
            labels={
                Config.COL_PRICE: 'Prezzo Totale (€, log)',
                Config.COL_PRICE_DAY: 'Costo Giornaliero (€/giorno, log)',
                Config.COL_AWARD: 'Criterio'
            },
            color_discrete_sequence=px.colors.qualitative.Set2
        )
        
        fig.update_layout(
            height=700, 
            showlegend=True,
            plot_bgcolor='rgba(248,249,250,1)',
            font=dict(family="Arial, sans-serif", size=12, color='#2C3E50')
        )
        fig.write_html(Config.PLOTS_DIR / '24_price_intensity_marginals.html')
        fig.show()
        
        # Ritorna statistiche per interpretazione esterna
        low_intensity = df_valid[df_valid[Config.COL_PRICE_DAY] < 1000]
        high_intensity = df_valid[df_valid[Config.COL_PRICE_DAY] > 10000]
        
        return {
            'total': len(df_valid),
            'low_intensity': len(low_intensity),
            'high_intensity': len(high_intensity),
            'low_pct': len(low_intensity)/len(df_valid)*100,
            'high_pct': len(high_intensity)/len(df_valid)*100
        }
    
    def generate_financial_metrics_table(self, df: pd.DataFrame):
        """Tabella metriche finanziarie aggregate per distretto (Pandas display)."""
        if Config.COL_DISTRICT not in df.columns or Config.COL_PRICE not in df.columns:
            print("Colonne necessarie non trovate")
            return None
        
        metrics = df.groupby(Config.COL_DISTRICT)[Config.COL_PRICE].agg([
            ('Valore_Totale', 'sum'),
            ('Valore_Medio', 'mean'),
            ('Valore_Mediano', 'median'),
            ('Num_Contratti', 'count'),
            ('Std_Dev', 'std')
        ]).reset_index()
        
        total_value = metrics['Valore_Totale'].sum()
        metrics['Quota_%'] = (metrics['Valore_Totale'] / total_value * 100).round(1)
        
        metrics = metrics.sort_values('Valore_Totale', ascending=False).head(15)
        
        # Formatta per display (converti in milioni)
        metrics_display = metrics.copy()
        metrics_display['Valore_Totale'] = (metrics_display['Valore_Totale']/1e6).round(1)
        metrics_display['Valore_Medio'] = (metrics_display['Valore_Medio']/1e6).round(2)
        metrics_display['Valore_Mediano'] = (metrics_display['Valore_Mediano']/1e6).round(2)
        metrics_display['Std_Dev'] = (metrics_display['Std_Dev']/1e6).round(2)
        
        # Rinomina colonne per display
        metrics_display.columns = ['Distretto', 'Valore Tot (€M)', 'Val Medio (€M)', 
                                   'Val Mediano (€M)', 'N. Contratti', 'Std Dev (€M)', 'Quota %']
        
        # Salva CSV
        metrics.to_csv(Config.PLOTS_DIR / 'metrics_by_district.csv', index=False)
        
        # Display con styling (compatibile nbconvert)
        print("\n=== Top 15 Distretti: Metriche Finanziarie Aggregate ===\n")
        display(metrics_display.style
                .background_gradient(subset=['Valore Tot (€M)'], cmap='Reds')
                .background_gradient(subset=['Quota %'], cmap='Blues')
                .format({'Valore Tot (€M)': '{:.1f}', 'Val Medio (€M)': '{:.2f}', 
                        'Val Mediano (€M)': '{:.2f}', 'Std Dev (€M)': '{:.2f}', 
                        'Quota %': '{:.1f}%'})
        )
        
        return metrics
    
    def plot_annual_volume_value_trend(self, df: pd.DataFrame):
        """Trend annuale dual-axis: volume (line) + valore (area) con Matplotlib."""
        if Config.COL_YEAR not in df.columns or Config.COL_PRICE not in df.columns:
            print("Colonne necessarie non trovate")
            return None
        
        yearly = df.groupby(Config.COL_YEAR).agg(
            Count=(Config.COL_YEAR, 'size'),
            Value=(Config.COL_PRICE, 'sum')
        ).reset_index()
        
        # Dual-axis matplotlib con colori migliorati
        fig, ax1 = plt.subplots(figsize=(14, 8))
        
        # Asse sinistro: Valore (area chart) - colore più tenue
        color1 = '#D73027'  # Rosso intenso ma non acceso
        ax1.set_xlabel('Anno', fontsize=13, weight='bold')
        ax1.set_ylabel('Valore Totale (€M)', fontsize=13, weight='bold', color=color1)
        ax1.fill_between(yearly[Config.COL_YEAR], yearly['Value']/1e6, 
                        alpha=0.25, color=color1, label='Valore Totale')
        ax1.plot(yearly[Config.COL_YEAR], yearly['Value']/1e6, 
                color=color1, linewidth=3, marker='o', markersize=9, 
                markerfacecolor='white', markeredgewidth=2.5, markeredgecolor=color1)
        ax1.tick_params(axis='y', labelcolor=color1, labelsize=11)
        ax1.grid(True, alpha=0.25, linestyle='--', color='gray')
        ax1.spines['top'].set_visible(False)
        
        # Asse destro: Volume (line chart) - blu più professionale
        ax2 = ax1.twinx()
        color2 = '#4575B4'  # Blu professionale
        ax2.set_ylabel('Numero Contratti', fontsize=13, weight='bold', color=color2)
        ax2.plot(yearly[Config.COL_YEAR], yearly['Count'], 
                color=color2, linewidth=3.5, marker='s', markersize=11, 
                markerfacecolor='white', markeredgewidth=2.5, markeredgecolor=color2,
                label='N. Contratti', linestyle='-')
        ax2.tick_params(axis='y', labelcolor=color2, labelsize=11)
        ax2.spines['top'].set_visible(False)
        
        # Titolo e legenda
        fig.suptitle("Trend Annuale: Volume vs Valore (Dual-Axis)", 
                    fontsize=16, weight='bold', color='#2C3E50')
        
        # Combina legende con frame
        lines1, labels1 = ax1.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', 
                  fontsize=12, frameon=True, fancybox=True, shadow=True)
        
        plt.tight_layout()
        plt.show()
        self._save(fig, '26_annual_volume_value_trend.png')
        
        # Calcola correlazione
        from scipy.stats import pearsonr
        corr, p_value = pearsonr(yearly['Count'], yearly['Value'])
        
        return {
            'correlation': corr,
            'p_value': p_value,
            'years': len(yearly)
        }

# --- INIZIALIZZAZIONE ANALYZER ---
comparative_analyzer = ComparativeAnalyzer()

### 6.1 Grafico 1: Scatter Plot Intensità Economica (Interattivo)

**Scelta Tecnica: Interactive Scatter Plot con Marginal Histograms**

Versione interattiva del density plot 4.4, implementa mantra Shneiderman completo in singola interfaccia.

**Design Space:**
* **Plot Centrale:** Scatter 2D log-log
  - X: Prezzo Totale, Y: Costo/Giorno
  - Colore: Criterio (0=blu scuro, 1=rosa, 2=giallo)
  - Punti individuali = contratti singoli
* **Marginal Plots:** Istogrammi distribuzioni univariate (overview contestuale)
* **Interattività:** 
  - Hover → details-on-demand (distretto, durata, valori esatti)
  - Zoom → focus geometrico su regioni
  - Click legenda → filter per criterio

**Implementazione Mantra:**
- **Overview:** Marginals + scatter completo
- **Zoom/Filter:** Interazione nativa Plotly
- **Details-on-demand:** Tooltip hover

---

**Interpretazione**

**Correlazione Positiva Dominante**

Traiettoria diagonale netta (pendenza ~1 in log-log):
- Prezzo totale aumenta → costo/giorno aumenta proporzionalmente
- Relazione power-law, non lineare
- Elasticità ~1: raddoppio prezzo ≈ raddoppio intensità

**Tre Bande di Intensità Identificate**

*Banda Bassa (<€1k/giorno, 714 contratti = 14.8%):*
- Base del grafico
- Progetti estesi a bassa intensità
- Economia di scala temporale

*Banda Standard (€1k-€10k/giorno, bulk ~80%):*
- Concentrazione principale
- Range normale intensità operativa
- Tutti i criteri rappresentati

*Banda Alta (>€10k/giorno, 270 contratti = 5.6%):*
- Top del grafico
- Progetti urgenti/intensivi
- Durata breve, alta concentrazione risorse

**Differenziazione per Criterio**

*Criterio 0 (blu scuro):*
- Concentrato centro-basso scatter
- Intensità prevalentemente standard
- Pattern coerente con bulk operativo

*Criterio 2 (giallo):*
- Dispersione maggiore
- Presenza significativa banda bassa E alta
- Più variabilità intra-criterio

*Criterio 1 (rosa):*
- Distribuzione intermedia
- Pattern simile a C0

**Outlier Mega-Progetti (interattivi)**

Hover rivela:
- Punti estremo destro: Beja mega-contratti
- Costo/giorno variabile (€5k-€50k)
- Nessun pattern univoco intensità per mega-opere

**Assenza Zona "Alto Prezzo + Bassa Intensità"**

Triangolo superiore destro vuoto:
- Nessun mega-progetto con €<500/giorno
- Limite tecnico: grandi opere richiedono intensità minima
- Conferma relazione strutturale prezzo-intensità

**Valore Interattività**

- Zoom su banda alta → identificazione progetti urgenti
- Filter criterio → confronto pattern intensità per metodo
- Hover outlier → contesto qualitativo (distretto, oggetto)

**Conclusione:** Scatter interattivo superiore a density plot per esplorazione dettagliata. Rivela correlazione power-law con tre regimi intensità. Interattività permette drill-down su anomalie (mega-progetti Beja, emergenze alta intensità) impossibile in visualizzazione statica aggregata.

In [ ]:
# --- ESECUZIONE SCATTER INTENSITÀ ---
stats_intensity = comparative_analyzer.plot_price_intensity_interactive(df_master)

if stats_intensity:
    print(f"Analisi Intensità Economica:")
    print(f"   • Contratti analizzati: {stats_intensity['total']:,}")
    print(f"   • Bassa intensità (<€1K/giorno): {stats_intensity['low_intensity']:,} ({stats_intensity['low_pct']:.1f}%)")
    print(f"   • Alta intensità (>€10K/giorno): {stats_intensity['high_intensity']:,} ({stats_intensity['high_pct']:.1f}%)")

### 6.2 Grafico 2: Tabella Metriche Finanziarie Aggregate per Distretto

**Scelta Tecnica: Tabella con Gradient Heatmap**

Tabella = forma definitiva di "Details-on-Demand" per lookup quantitativo preciso.

**Design Space:**
* **Righe:** Distretti (ordinati per Valore Totale decrescente)
* **Colonne:** 7 metriche quantitative
* **Gradient:** Saturazione colore = intensità valore (rosso scuro = max)
* **Formato:** Valori numerici esatti con precisione decimale

**Motivazione - Complementarietà Tabella/Grafico:**

Grafici ottimizzati per **percezione pattern** (trend, outlier, confronti relativi). Tabelle ottimizzate per **cognizione analitica** (lookup esatto, calcoli mentali, verifica). Ridondanza dati (Borkin et al.): mostrare stessi valori in forma visuale (mappe/bar) E testuale (tabella) migliora comprensione e retention.

---

**Interpretazione**

**Conferma Anomalia Beja (Riga 1)**

- **Valore Totale:** €590.6M (15.3% mercato totale, 1.2x Porto+Lisboa)
- **Valore Medio:** €4.25M (6x mediana, 5.6x Porto)
- **N. Contratti:** 139 (solo 3% del totale)
- **Std Dev:** €7.89M (altissima variabilità)
- **Insight:** Pochi mega-progetti con variabilità estrema

**Top 3 = 39.1% Concentrazione**

- Beja (15.3%) + Porto (12.4%) + Lisboa (11.4%) = 39.1%
- Livello concentrazione: MEDIA (non oligopolio né frammentazione totale)
- Pattern coerente con economie sviluppate

**Inversione Ranking Valore Totale vs Medio**

*Valore Totale (colonna 2):*
1. Beja, 2. Porto, 3. Lisboa

*Valore Medio (colonna 3):*
1. Beja (€4.25M), 2. Madeira (€2.24M), 3. Açores (€0.91M)

Metropoli (Porto/Lisboa) con valori medi bassi (€0.66-0.69M) confermano frammentazione operativa.

**Mediana vs Media: Asimmetria Distribuzioni**

Tutti i distretti: Valore Medio >> Valore Mediano
- Beja: €4.25M medio vs €0.76M mediano (5.6x)
- Indica distribuzione right-skewed con outlier estremi
- Conferma power-law identificata in analisi prezzi

**Standard Deviation: Volatilità**

*Alta variabilità (Std Dev >€5M):*
- Beja (€7.89M), Vila Real (€7.22M), Madeira (€6.45M)
- Mercati con mix estremo: micro-appalti + mega-progetti

*Bassa variabilità (Std Dev <€1M):*
- Faro (€0.54M), Coimbra (€0.63M), Santarém (€0.92M)
- Mercati omogenei, appalti standardizzati

**Long Tail: Distretti 6-15**

Quota individuale 2.6-5.3%, combinati = ~40%
Nessun distretto irrilevante: mercato distribuito geograficamente

**Gradient Heatmap: Percezione Preattentiva**

Rosso scuro (Beja riga 1) identificabile istantaneamente senza lettura numeri. Combina velocità percezione visiva + precisione cognitiva testuale.

**Conclusione:** Tabella sintetizza quantitativamente tutti gli insight visuali precedenti. Beja: anomalia statistica confermata su tutte le metriche. Metropoli: alto volume ma basso valore medio. Gradient trasforma tabella da strumento lookup a strumento pattern recognition, unendo forze tabelle e grafici.

In [ ]:
# --- ESECUZIONE TABELLA METRICHE ---
metrics_df = comparative_analyzer.generate_financial_metrics_table(df_master)

if metrics_df is not None:
    top3_share = metrics_df.head(3)['Quota_%'].sum()
    print(f"\nConcentrazione Mercato:")
    print(f"   • Top 3 distretti = {top3_share:.1f}% valore totale")
    print(f"   • Livello concentrazione: {'ALTA (oligopolio)' if top3_share > 50 else 'MEDIA' if top3_share > 35 else 'BASSA (frammentato)'}")

### 6.3 Grafico 3: Trend Annuale Volume vs Valore (Dual-Axis)

**Scelta Tecnica: Dual-Axis Chart con Area+Line**

Confronta due variabili quantitative con scale incompatibili su substrato temporale condiviso.

**Design Space:**
* **Asse X:** Anno (Quantitativo temporale)
* **Asse Y Sinistro:** Valore Totale €M (scala 0-700M)
* **Asse Y Destro:** N. Contratti (scala 0-800)
* **Segno 1:** Area (rosso trasparente) = massa economica
* **Segno 2:** Linea con marker (blu) = trend volumetrico

Dual-axis controverso (può distorcere percezione) ma necessario per sovrapposizione temporale diretta. Uso segni diversi (area vs linea) riduce confusione visiva.

---

**Interpretazione**

**Fase 1 (2010-2014): Crescita Asincrona**

*Volume (blu):*
- Crescita esponenziale: 0 → 450 contratti (+450 in 4 anni)
- Accelerazione costante

*Valore (rosso):*
- Crescita super-esponenziale: €30M → €700M (+2233%)
- Picco 2014 = apice storico assoluto

**Decoupling 2014:**
- Volume a metà plateau (450), valore al massimo (€700M)
- Mega-progetti concentrati: pochi contratti ad altissimo valore
- Beja-effect: outlier dominano aggregato

**Fase 2 (2014-2015): Shock Asimmetrico**

*Valore: crollo verticale -60% (€700M → €280M)*
*Volume: calo moderato -30% (450 → 340)*

**Causa:** Discontinuità normativa 2014-2015 (abolizione Criterio 0). Stop nuovi bandi mega-progetti, esecuzione contratti esistenti prosegue → volume più resiliente.

**Fase 3 (2016-2020): Ripresa Biforcata**

*2016-2017:*
- Volume: recovery parziale (340 → 430)
- Valore: stagnazione bassa (€270-300M)
- Pattern opposto a 2010-2014: molti contratti piccoli

*2017-2020:*
- Volume: crescita costante → picco 780 (2020, +73% vs 2014)
- Valore: recovery moderata → €460M (2017), plateau €370-390M
- **Elasticità rotta:** volume massimo storico NON corrisponde a valore massimo

**Decoupling 2020:**
- Massimo volume (780) + valore medio-basso (€370M)
- Frammentazione estrema: €475k medio per contratto
- Opposto di 2014: tanti appalti piccoli vs pochi giganti

**Fase 4 (2021-2022): Collasso Totale**

Entrambe le metriche → 0
- Dati incompleti O fine dataset O shock COVID ritardato

**Correlazione vs Causalità**

Correlazione generale positiva (r>0.7 stimato visivamente) MA:
- Non proporzionale: elasticità variabile nel tempo
- Anni anomali (2014, 2020) rompono relazione
- Eventi esterni (normative, cicli economici) dominano su correlazione intrinseca

**Pattern Ciclico Assente**

Nessuna periodicità: no cicli elettorali quadriennali evidenti. Discontinuità discrete dominano su trend graduali.

**Conclusione:** Dual-axis rivela relazione volume-valore non stabile. 2014 = era mega-progetti (alto valore/basso volume), 2020 = era frammentazione (alto volume/basso valore). Crollo 2015 conferma shock normativo come turning point. Serie temporale mostra struttura a-ciclica guidata da eventi discreti, non da trend macroeconomici graduali.

In [ ]:
# --- ESECUZIONE TREND ANNUALE ---
trend_stats = comparative_analyzer.plot_annual_volume_value_trend(df_master)

if trend_stats:
    print(f"\nAnalisi Correlazione Volume-Valore:")
    print(f"   • Coefficiente correlazione Pearson: r = {trend_stats['correlation']:.3f}")
    print(f"   • Significatività statistica: p-value = {trend_stats['p_value']:.4f}")
    print(f"   • Anni analizzati: {trend_stats['years']}")
    
    if trend_stats['p_value'] < 0.05:
        strength = 'FORTE' if abs(trend_stats['correlation']) > 0.7 else 'MODERATA' if abs(trend_stats['correlation']) > 0.4 else 'DEBOLE'
        print(f"   • Interpretazione: Correlazione {strength} e statisticamente significativa")
        print(f"   • Implicazione: Mercato {'elastico - crescita economica si traduce in più contratti E più valore' if trend_stats['correlation'] > 0 else 'anelastico - dinamiche volume/valore disaccoppiate'}")

## 7. Insight Strategici Finali

**1. Discontinuità 2014-2015: Shock Strutturale**

- Criterio 0 (lowest price) crolla: valore -60%, volume -30%
- Causa incerta: possibile cambio normativo, fine ciclo fondi EU, o shift mercato
- Transizione a Criteri 1/2: mercato resta asimmetrico fino 2022
- Conseguenza: frammentazione crescente (picco volume 2020, valore medio minimo storico)

**2. Beja: Anomalia Statistica Persistente**

- 15% valore totale, 3% volume → €4.25M medio (6x mediana)
- Concentrazione pre-2014 su mega-progetti mai replicati
- Geografia dual: metropoli = frammentazione operativa, periferie = hub strategici

**3. Decoupling Volume-Valore: Elasticità Variabile**

- 2014: alto valore/basso volume (era mega-progetti)
- 2020: alto volume/basso valore (era frammentazione)
- Assenza correlazione stabile → mercato guidato da shock discreti

**4. Distribuzione Power-Law con Bimodalità**

- Bulk: €100k-€500k, 100-300 giorni (60% mercato)
- Outlier: >€10M, >1 anno (<5% volume, >30% valore)
- Durata = proxy affidabile per complessità/valore

**5. Omogeneità Linguistica, Specializzazione Geografica**

- Vocabolario standardizzato indipendente da criterio
- Assenza linguaggio strategico differenziato
- Clustering: dominanza "lavori civili generici" (80%)

**6. Assenza Stagionalità Ricorrente**

- Variabilità anno >> variabilità mensile
- Pattern driven da cicli pluriennali, non fattori stagionali

**Implicazioni:**

- **Operatori:** Geo-specializzazione critica, durata predice pricing
- **Policy maker:** Discontinuità 2014 richiede indagine
- **Analisti:** Visualizzazione multi-scala essenziale vs statistiche aggregate

**Metodologia validata:** Viste multiple + interattività + ridondanza dati hanno rivelato pattern invisibili in analisi univariata.

In [ ]:
!jupyter nbconvert --to html --no-input --template lab preProcessData_story_ref.ipynb